In [ ]:
## u22 urdf
import numpy as np
import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# ====== 你要改的参数 ======
urdf_right_path = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-right_arm/urdf/V1.urdf"
urdf_left_path  = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-left_arm/urdf/V5.25.0210.urdf"

base_pos_right = [0.0, -0.25, 0.0]
base_pos_left  = [0.0,  0.25, 0.0]
base_orn_right = [0, 0, 0]
base_orn_left  = [0, 0, 0]

# 多帧关节角：list of list
# T = 50
# joint_angles_right = [[0.02*i, 0, 0, 0, 0, 0, 0] for i in range(T)]
# joint_angles_left  = [[-0.02*i, 0, 0, 0, 0, 0, 0] for i in range(T)]

json_path = "/liujinxin/dataset/bimanual/tidy_tools_filtered/1125_bimanual_long/take/pliers/data.json"
with open(json_path, "r") as f:
    data = json.load(f)

joint_angles_right = []
joint_angles_left = []

for step in data:
    state = data["joint_angles"]
    joint_right = state[:7]
    joint_left = state[7:-2]
    joint_angles_right.append(joint_right)
    joint_angles_left.append(joint_left)




# =========================


def set_joint_states(robot_id: int, angles):
    for i, a in enumerate(angles):
        p.resetJointState(robot_id, i, float(a))

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
    return points, parent

def compute_frame(robot_id_right, robot_id_left, f):
    set_joint_states(robot_id_right, joint_angles_right[f])
    set_joint_states(robot_id_left,  joint_angles_left[f])
    pts_r, parent_r = get_links_world(robot_id_right)
    pts_l, parent_l = get_links_world(robot_id_left)
    return pts_r, parent_r, pts_l, parent_l

def skeleton_arrays(points, parent):
    # lines arrays with None separators
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        pidx = parent[i]
        if pidx >= 0:
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def set_scene_ranges(fig, pts_r, pts_l):
    all_pts = np.vstack([pts_r, pts_l])
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2.0

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
            yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
            zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
            aspectmode="cube",
        )
    )

# ====== PyBullet 初始化（Notebook 里只做一次） ======
# 避免重复 connect
try:
    cid = p.getConnectionInfo()['isConnected']
except Exception:
    cid = False

if not cid:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

robot_id_right = p.loadURDF(
    urdf_right_path,
    basePosition=base_pos_right,
    baseOrientation=p.getQuaternionFromEuler(base_orn_right),
    useFixedBase=True
)
robot_id_left = p.loadURDF(
    urdf_left_path,
    basePosition=base_pos_left,
    baseOrientation=p.getQuaternionFromEuler(base_orn_left),
    useFixedBase=True
)

# ====== 初始帧 ======
f0 = 0
pts_r, parent_r, pts_l, parent_l = compute_frame(robot_id_right, robot_id_left, f0)

xr, yr, zr = skeleton_arrays(pts_r, parent_r)
xl, yl, zl = skeleton_arrays(pts_l, parent_l)

# ====== Plotly FigureWidget（关键：可交互 + 可更新） ======
fig = go.FigureWidget()

# 右臂 points
fig.add_trace(go.Scatter3d(
    x=pts_r[:,0], y=pts_r[:,1], z=pts_r[:,2],
    mode="markers",
    marker=dict(size=4),
    name="right_points"
))
# 右臂 bones
fig.add_trace(go.Scatter3d(
    x=xr, y=yr, z=zr,
    mode="lines",
    line=dict(width=4),
    name="right_bones"
))

# 左臂 points
fig.add_trace(go.Scatter3d(
    x=pts_l[:,0], y=pts_l[:,1], z=pts_l[:,2],
    mode="markers",
    marker=dict(size=4),
    name="left_points"
))
# 左臂 bones
fig.add_trace(go.Scatter3d(
    x=xl, y=yl, z=zl,
    mode="lines",
    line=dict(width=4),
    name="left_bones"
))

fig.update_layout(
    title=f"FK 3D Viewer (Frame {f0}) — drag to rotate / scroll to zoom / right-drag to pan",
    margin=dict(l=0, r=0, t=40, b=0),
    height=700,
    legend=dict(orientation="h")
)
set_scene_ranges(fig, pts_r, pts_l)

# ====== Slider + 回调（更新数据，不重画） ======
slider = widgets.IntSlider(
    value=f0, min=0, max=T-1, step=1,
    description="frame", continuous_update=False
)

def on_slider_change(change):
    f = int(change["new"])
    pts_r, parent_r, pts_l, parent_l = compute_frame(robot_id_right, robot_id_left, f)

    xr, yr, zr = skeleton_arrays(pts_r, parent_r)
    xl, yl, zl = skeleton_arrays(pts_l, parent_l)

    with fig.batch_update():
        # right_points
        fig.data[0].x = pts_r[:,0]; fig.data[0].y = pts_r[:,1]; fig.data[0].z = pts_r[:,2]
        # right_bones
        fig.data[1].x = xr;         fig.data[1].y = yr;         fig.data[1].z = zr
        # left_points
        fig.data[2].x = pts_l[:,0]; fig.data[2].y = pts_l[:,1]; fig.data[2].z = pts_l[:,2]
        # left_bones
        fig.data[3].x = xl;         fig.data[3].y = yl;         fig.data[3].z = zl

        fig.layout.title = f"FK 3D Viewer (Frame {f}) — drag to rotate / scroll to zoom / right-drag to pan"
        set_scene_ranges(fig, pts_r, pts_l)

slider.observe(on_slider_change, names="value")

display(slider)
display(fig)


pybullet build time: Jan 29 2025 23:16:28


NameError: name 'json' is not defined

In [ ]:
## test
1

1

In [2]:
import os, math, shutil, json, asyncio, traceback
import numpy as np
import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 你只需要主要改这里：CAMERAS（每个相机自己的 w/h/fx/fy/cx/cy）
# ============================================================
CAMERAS = {
    # 你说 hand 是 640x480：把 hand_0/hand_1 改成这样即可
    "hand_0": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),   # cx/cy None => 自动 w/2,h/2
    "hand_1": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "head":   dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    """统一入口：所有相机参数从这里拿；cx/cy 若未填则自动置中"""
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([[s["fx"], 0.0,   s["cx"]],
                  [0.0,     s["fy"], s["cy"]],
                  [0.0,     0.0,    1.0]], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ===================== 机器人 & 任务参数区（按需改） =====================
urdf_right_path = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-right_arm/urdf/V1.urdf"
urdf_left_path  = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-left_arm/urdf/V5.25.0210.urdf"

base_pos_right = [0.0, -0.25, 0.0]
base_pos_left  = [0.0,  0.25, 0.0]
base_orn_right = [0, 0, 0]
base_orn_left  = [0, 0, 0]

wrist_link_idx_left  = 6
wrist_link_idx_right = 6

# wrist camera 相对 wrist 的安装（cam 坐标：默认朝 +X）
CAM_OFFSET_M = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

# head camera 固定：两臂中点 + 高 30cm，俯视 30°
HEAD_HEIGHT_M = 0.30
HEAD_PITCH_DOWN_DEG = -30.0

# 相机盒子尺寸（米）：长x宽x高（局部 X/Y/Z）
CAM_BOX_LWH = (0.06, 0.03, 0.03)

# 关节角数据
# json_path = "/liujinxin/dataset/bimanual/tidy_tools_filtered/1125_bimanual_long/take/pliers/data.json"
json_path = "/liujinxin/code/tram/GR00T-Dreams-main/data_json_test.json"
with open(json_path, "r") as f:
    data = json.load(f)

T = len(data)
joint_angles_right, joint_angles_left = [], []
for step in data:
    state = step["joint_angles"]
    joint_angles_right.append(state[:7])
    joint_angles_left.append(state[7:-2])

# 导出目录
EXPORT_DIR = "export_fk"
FPS = 10

# 导出渲染采用哪个相机的宽高比（只影响导出视频/动图的“画面比例”）
EXPORT_RENDER_CAM = "head"   # 你也可以改成 "hand_0" / "hand_1"

# 是否保存相机 txt（intrinsic + 每帧 extrinsic）
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"
# ======================================================================


# ===================== 数学/几何工具 =====================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0],
                     [0, ca, -sa],
                     [0, sa, ca]], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[ ca, 0, sa],
                     [  0, 1,  0],
                     [-sa, 0, ca]], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    """
    约定：cam 的“朝前”是 +X；俯视用绕 Y 旋转（你之前也在用 rot_y）
    """
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    mid = (np.array(base_pos_left, dtype=np.float64) + np.array(base_pos_right, dtype=np.float64)) / 2.0
    pos = mid + np.array([0.0, 0.0, HEAD_HEIGHT_M], dtype=np.float64)
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L/2; y = W/2; z = H/2
    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)
    R = T_world_obj[:3,:3]
    t = T_world_obj[:3,3]
    corners_world = (R @ corners_local.T).T + t[None,:]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a,b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:,0]*axis_len
    y_end = o + R[:,1]*axis_len
    z_end = o + R[:,2]*axis_len
    return o, x_end, y_end, z_end


# ===================== PyBullet FK =====================
def set_joint_states(robot_id: int, angles):
    for i, a in enumerate(angles):
        p.resetJointState(robot_id, i, float(a))

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])
    return points, parent, R_list

def compute_frame(robot_id_right, robot_id_left, f):
    set_joint_states(robot_id_right, joint_angles_right[f])
    set_joint_states(robot_id_left,  joint_angles_left[f])
    pts_r, parent_r, Rr = get_links_world(robot_id_right)
    pts_l, parent_l, Rl = get_links_world(robot_id_left)
    return pts_r, parent_r, Rr, pts_l, parent_l, Rl

def skeleton_lines(points, parent):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        pidx = parent[i]
        if pidx >= 0:
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(pts_r, pts_l, cams_T_dict):
    all_pts = [pts_r, pts_l]
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3,3][None,:])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs)/2
    span = (maxs - mins).max()
    if span < 1e-6: span = 1.0
    half = span/2
    return center, half


# ===================== 初始化 PyBullet（只一次） =====================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

robot_id_right = p.loadURDF(
    urdf_right_path,
    basePosition=base_pos_right,
    baseOrientation=p.getQuaternionFromEuler(base_orn_right),
    useFixedBase=True
)
robot_id_left = p.loadURDF(
    urdf_left_path,
    basePosition=base_pos_left,
    baseOrientation=p.getQuaternionFromEuler(base_orn_left),
    useFixedBase=True
)

T_world_head = head_cam2world_fixed()


# ===================== Plotly FigureWidget + UI =====================
f0 = 0
pts_r, parent_r, Rr, pts_l, parent_l, Rl = compute_frame(robot_id_right, robot_id_left, f0)

def cams_T_from_frame(pts_r, Rr, pts_l, Rl):
    return {
        "hand_0": cam2world_from_link(pts_l, Rl, wrist_link_idx_left, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "hand_1": cam2world_from_link(pts_r, Rr, wrist_link_idx_right, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":   T_world_head,
    }

cams_T = cams_T_from_frame(pts_r, Rr, pts_l, Rl)

xr, yr, zr = skeleton_lines(pts_r, parent_r)
xl, yl, zl = skeleton_lines(pts_l, parent_l)

fig = go.FigureWidget()
fig.add_trace(go.Scatter3d(x=pts_r[:,0], y=pts_r[:,1], z=pts_r[:,2], mode="markers",
                           marker=dict(size=4), name="right_points"))
fig.add_trace(go.Scatter3d(x=xr, y=yr, z=zr, mode="lines", line=dict(width=4), name="right_bones"))
fig.add_trace(go.Scatter3d(x=pts_l[:,0], y=pts_l[:,1], z=pts_l[:,2], mode="markers",
                           marker=dict(size=4), name="left_points"))
fig.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode="lines", line=dict(width=4), name="left_bones"))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
                               mode="lines", line=dict(width=6, color="red"),
                               name=f"{prefix}_x", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
                               mode="lines", line=dict(width=6, color="green"),
                               name=f"{prefix}_y", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
                               mode="lines", line=dict(width=6, color="blue"),
                               name=f"{prefix}_z", visible=visible, showlegend=False))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x,y,z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(width=4, color="purple"),
                               name=f"{prefix}_box", visible=visible, showlegend=False))

trace_map = {
    "right_points": [0],
    "right_bones": [1],
    "left_points": [2],
    "left_bones": [3],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

# world axes
base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx+3))

# camera axes + boxes
for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx+3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges(pts_r, pts_l, cams_T)
fig.update_layout(
    title=f"FK + Cameras (Frame {f0}) — drag/zoom/pan",
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)

# --------- UI controls ----------
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_right = widgets.Checkbox(value=True, description="Right arm")
cb_show_left  = widgets.Checkbox(value=True, description="Left arm")
cb_show_world = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider])
ui_row2 = widgets.HBox([cb_show_right, cb_show_left, cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row3 = widgets.HBox([cam_select, dd_export, btn_export])


# --------- 播放控件（asyncio，避免你之前 thread 报错） ----------
btn_play  = widgets.Button(description="Play ▶", button_style="success")
btn_pause = widgets.Button(description="Pause ⏸", button_style="warning")
btn_step  = widgets.Button(description="Step +1", button_style="")
speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")
ui_row_play = widgets.HBox([btn_play, btn_pause, btn_step, speed, cb_loop])

_play_state = {"running": False, "task": None}

async def _player_loop_async():
    try:
        while _play_state["running"]:
            cur = int(slider.value)
            nxt = cur + 1
            if nxt >= T:
                if cb_loop.value:
                    nxt = 0
                else:
                    _play_state["running"] = False
                    break

            slider.value = nxt
            dt = 1.0 / max(1, int(speed.value))
            await asyncio.sleep(dt)

    except asyncio.CancelledError:
        pass
    except Exception:
        with out_log:
            print("[play][ERROR]")
            traceback.print_exc()
    finally:
        _play_state["running"] = False
        _play_state["task"] = None

def on_play_clicked(_):
    if _play_state["running"]:
        return
    _play_state["running"] = True
    _play_state["task"] = asyncio.create_task(_player_loop_async())
    with out_log:
        print("[play] started")

def on_pause_clicked(_):
    _play_state["running"] = False
    task = _play_state.get("task")
    if task is not None:
        task.cancel()
    with out_log:
        print("[play] paused")

def on_step_clicked(_):
    cur = int(slider.value)
    if cur < T - 1:
        slider.value = cur + 1
    elif cb_loop.value:
        slider.value = 0

btn_play.on_click(on_play_clicked)
btn_pause.on_click(on_pause_clicked)
btn_step.on_click(on_step_clicked)


def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        set_visible(trace_map["right_points"], cb_show_right.value)
        set_visible(trace_map["right_bones"], cb_show_right.value)
        set_visible(trace_map["left_points"], cb_show_left.value)
        set_visible(trace_map["left_bones"], cb_show_left.value)
        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)

def update_frame(f):
    f = int(f)
    pts_r, parent_r, Rr, pts_l, parent_l, Rl = compute_frame(robot_id_right, robot_id_left, f)

    xr, yr, zr = skeleton_lines(pts_r, parent_r)
    xl, yl, zl = skeleton_lines(pts_l, parent_l)

    cams_T_new = cams_T_from_frame(pts_r, Rr, pts_l, Rl)
    center, half = scene_ranges(pts_r, pts_l, cams_T_new)

    with fig.batch_update():
        fig.data[0].x = pts_r[:,0]; fig.data[0].y = pts_r[:,1]; fig.data[0].z = pts_r[:,2]
        fig.data[1].x = xr;         fig.data[1].y = yr;         fig.data[1].z = zr
        fig.data[2].x = pts_l[:,0]; fig.data[2].y = pts_l[:,1]; fig.data[2].z = pts_l[:,2]
        fig.data[3].x = xl;         fig.data[3].y = yl;         fig.data[3].z = zl

        idx = 4
        idx += 3  # skip world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)
            fig.data[idx+0].x = [o[0], x_end[0]]; fig.data[idx+0].y = [o[1], x_end[1]]; fig.data[idx+0].z = [o[2], x_end[2]]
            fig.data[idx+1].x = [o[0], y_end[0]]; fig.data[idx+1].y = [o[1], y_end[1]]; fig.data[idx+1].z = [o[2], y_end[2]]
            fig.data[idx+2].x = [o[0], z_end[0]]; fig.data[idx+2].y = [o[1], z_end[1]]; fig.data[idx+2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx; fig.data[idx].y = by; fig.data[idx].z = bz
            idx += 1

        fig.layout.title = f"FK + Cameras (Frame {f}) — drag/zoom/pan"
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()

def on_slider_change(change):
    update_frame(change["new"])

slider.observe(on_slider_change, names="value")

for w in [cb_show_right, cb_show_left, cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()


# ===================== Matplotlib 渲染（导出用） =====================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    pts_r, parent_r, Rr, pts_l, parent_l, Rl = compute_frame(robot_id_right, robot_id_left, f)
    cams_T = cams_T_from_frame(pts_r, Rr, pts_l, Rl)

    # ✅ 按相机 w/h 自动设置画面比例（只需改 CAMERAS）
    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(pts_r[:,0], pts_r[:,1], pts_r[:,2], s=10)
    for i, pidx in enumerate(parent_r):
        if pidx >= 0:
            ax.plot([pts_r[pidx,0], pts_r[i,0]],
                    [pts_r[pidx,1], pts_r[i,1]],
                    [pts_r[pidx,2], pts_r[i,2]], c="k")

    ax.scatter(pts_l[:,0], pts_l[:,1], pts_l[:,2], s=10)
    for i, pidx in enumerate(parent_l):
        if pidx >= 0:
            ax.plot([pts_l[pidx,0], pts_l[i,0]],
                    [pts_l[pidx,1], pts_l[i,1]],
                    [pts_l[pidx,2], pts_l[i,2]], c="k")

    # 相机坐标轴
    for Tm in cams_T.values():
        o = Tm[:3,3]; R = Tm[:3,:3]
        ax.quiver(o[0],o[1],o[2], R[0,0],R[1,0],R[2,0], color="r", length=0.08)
        ax.quiver(o[0],o[1],o[2], R[0,1],R[1,1],R[2,1], color="g", length=0.08)
        ax.quiver(o[0],o[1],o[2], R[0,2],R[1,2],R[2,2], color="b", length=0.08)

    all_pts = np.vstack([pts_r, pts_l, np.stack([T[:3,3] for T in cams_T.values()])])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2
    span = (maxs - mins).max()
    if span < 1e-6: span = 1.0
    half = span / 2

    ax.set_xlim(center[0]-half, center[0]+half)
    ax.set_ylim(center[1]-half, center[1]+half)
    ax.set_zlim(center[2]-half, center[2]+half)
    ax.set_box_aspect([1,1,1])
    ax.axis("off")
    ax.set_title(f"FK + Cameras (Frame {f})")

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig)
    return img, cams_T


# ===================== 保存相机 txt（可选） =====================
def maybe_save_camera_txt(cams_T: dict, frame_idx: int):
    if not SAVE_CAM_TXT:
        return
    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)

    # intrinsic（每个相机一份）
    for name in CAMERAS.keys():
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")

    # extrinsic：这里保存 cam2world 4x4；你需要 world2cam 的话再取 inverse
    for name, T_wc in cams_T.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), T_wc, fmt="%.9f")


# ===================== 导出（gif/mp4） =====================
def export_frames(export_type: str):
    os.makedirs(EXPORT_DIR, exist_ok=True)

    with out_log:
        clear_output()
        print(f"[export] type={export_type}")
        print("[export] renderer = matplotlib(Agg)")
        print("[export] render_aspect_from =", EXPORT_RENDER_CAM, cam_spec(EXPORT_RENDER_CAM))

    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk.gif")
        frames = []
        for f in range(T):
            img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
            frames.append(img)
            maybe_save_camera_txt(cams_T, f)
            if f % 10 == 0:
                with out_log:
                    print(f"[export] frame {f}/{T-1}")
        imageio.mimsave(out_path, frames, duration=1.0 / FPS)
        with out_log:
            print(f"[export] GIF saved: {out_path}")

    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk.mp4")
        ffmpeg_bin = shutil.which("ffmpeg")
        if ffmpeg_bin is None:
            with out_log:
                print("[export][ERROR] ffmpeg not found. Install: sudo apt-get update && sudo apt-get install -y ffmpeg")
            return

        writer = imageio.get_writer(
            out_path,
            fps=FPS,
            codec="libx264",
            format="FFMPEG",
            ffmpeg_params=["-pix_fmt", "yuv420p"],
        )
        try:
            for f in range(T):
                img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
                writer.append_data(img)
                maybe_save_camera_txt(cams_T, f)
                if f % 10 == 0:
                    with out_log:
                        print(f"[export] frame {f}/{T-1}")
        finally:
            writer.close()

        with out_log:
            print(f"[export] MP4 saved: {out_path}")

    else:
        with out_log:
            print(f"[export] unknown export_type={export_type}")

def on_export_clicked(_):
    typ = dd_export.value
    with out_log:
        clear_output()
    if typ == "none":
        with out_log:
            print("请选择导出格式 gif 或 mp4")
        return
    export_frames(typ)

btn_export.on_click(on_export_clicked)


# ===================== 展示 =====================
display(ui_row1, ui_row_play, ui_row2, ui_row3, out_log, fig)


pybullet build time: Jan 29 2025 23:16:28


Output()

FigureWidget({
    'data': [{'marker': {'size': 4},
              'mode': 'markers',
              'name': 'right_points',
              'type': 'scatter3d',
              'uid': 'cf419332-ed6a-49b0-9bca-03615426d334',
              'visible': True,
              'x': {'bdata': ('AAAAAAAAwL0AAAAAAAAQPgAAAICi0Z' ... 'AAAJUjwD8AAACgRKbMPwAAAADVVM8/'),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAIFyP2r8AAABAZmbevwAAAKCDXu' ... 'AAILgc4L8AAABADfTavwAAAOAONd2/'),
                    'dtype': 'f8'},
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAwPgAAAICy17' ... 'AAgKAs0L8AAADAYCjPvwAAAMBv/c6/'),
                    'dtype': 'f8'}},
             {'line': {'width': 4},
              'mode': 'lines',
              'name': 'right_bones',
              'type': 'scatter3d',
              'uid': '5fe32620-38dd-45b8-baca-a66f42b488b9',
              'visible': True,
              'x': array([-2.9103830456733704e-11, 9.313225746154785e-10, None,
                        

In [43]:
import os, math, shutil, json, asyncio, traceback
import numpy as np
import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 相机参数
# ============================================================
CAMERAS = {
    "hand_0": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "hand_1": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "head":   dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([[s["fx"], 0.0,   s["cx"]],
                  [0.0,     s["fy"], s["cy"]],
                  [0.0,     0.0,    1.0]], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ===================== 机器人 & 任务参数区（按需改） =====================
urdf_right_path = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-right_arm/urdf/V1.urdf"
urdf_left_path  = "/liujinxin/code/tram/cosmos-predict2.5/outputs/u22/V1-left_arm/urdf/V5.25.0210.urdf"

# 原始 pred 位置
base_pos_right = [0.0, -0.25, 0.0]
base_pos_left  = [0.0,  0.25, 0.0]
base_orn_right = [0, 0, 0]
base_orn_left  = [0, 0, 0]

# ============================================================
# ✅ 用于区分 pred/target 两套机器人
# target 会整体沿 Y 轴平移这个距离（单位：米）
# 设成 0.0 表示不人为错开
# ============================================================
COMPARE_OFFSET_Y_M = 0.

wrist_link_idx_left  = 6
wrist_link_idx_right = 6

CAM_OFFSET_M = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

HEAD_HEIGHT_M = 0.30
HEAD_PITCH_DOWN_DEG = -30.0

CAM_BOX_LWH = (0.06, 0.03, 0.03)

# ============================================================
# ✅ 两条 action 路径：pred + target
# ============================================================
# json_path_pred = "/liujinxin/code/tram/GR00T-Dreams-main/output/test/0/data_json_test.json"
# json_path_target = "/liujinxin/code/tram/GR00T-Dreams-main/output/test/0/data_target_json_test.json"

EXPORT_DIR = "export_fk"
FPS = 10

EXPORT_RENDER_CAM = "head"
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"
# ======================================================================


# ===================== 读取 / padding action 工具 =====================
def load_action_json(json_path, key="joint_angles"):
    with open(json_path, "r") as f:
        data = json.load(f)

    actions = []
    for step in data:
        if key not in step:
            raise KeyError(f"{json_path} 里没有 key='{key}'")
        actions.append(step[key])

    arr = np.asarray(actions, dtype=np.float64)
    if arr.ndim != 2:
        raise ValueError(f"{json_path} 读取后 action shape 异常: {arr.shape}")
    return arr

def pad_array_to_shape(arr, target_len=None, target_dim=None, pad_mode="edge"):
    arr = np.asarray(arr, dtype=np.float64)
    if arr.ndim != 2:
        raise ValueError(f"arr must be 2D, got shape={arr.shape}")

    T0, D0 = arr.shape
    if target_len is None:
        target_len = T0
    if target_dim is None:
        target_dim = D0

    if D0 < target_dim:
        if pad_mode == "edge":
            if D0 == 0:
                arr = np.zeros((T0, target_dim), dtype=np.float64)
            else:
                pad_cols = np.repeat(arr[:, -1:], target_dim - D0, axis=1)
                arr = np.concatenate([arr, pad_cols], axis=1)
        elif pad_mode == "zero":
            pad_cols = np.zeros((T0, target_dim - D0), dtype=np.float64)
            arr = np.concatenate([arr, pad_cols], axis=1)
        else:
            raise ValueError(f"unknown pad_mode={pad_mode}")
    elif D0 > target_dim:
        arr = arr[:, :target_dim]

    T1 = arr.shape[0]
    if T1 < target_len:
        if pad_mode == "edge":
            if T1 == 0:
                pad_rows = np.zeros((target_len, arr.shape[1]), dtype=np.float64)
                arr = pad_rows
            else:
                pad_rows = np.repeat(arr[-1:, :], target_len - T1, axis=0)
                arr = np.concatenate([arr, pad_rows], axis=0)
        elif pad_mode == "zero":
            pad_rows = np.zeros((target_len - T1, arr.shape[1]), dtype=np.float64)
            arr = np.concatenate([arr, pad_rows], axis=0)
        else:
            raise ValueError(f"unknown pad_mode={pad_mode}")
    elif T1 > target_len:
        arr = arr[:target_len]

    return arr

def split_bimanual_actions(action_arr):
    """
    默认假设:
      right = 前7维
      left  = 第7维到倒数第2维
    """
    joint_angles_right = []
    joint_angles_left = []
    for state in action_arr:
        joint_angles_right.append(state[:7])
        joint_angles_left.append(state[7:-2])
    return np.asarray(joint_angles_right, dtype=np.float64), np.asarray(joint_angles_left, dtype=np.float64)


# ===================== 读取 pred / target =====================
action_pred_raw = load_action_json(json_path_pred, key="joint_angles")
action_target_raw = load_action_json(json_path_target, key="joint_angles")

T_pred_raw, D_pred_raw = action_pred_raw.shape
T_target_raw, D_target_raw = action_target_raw.shape

T = max(T_pred_raw, T_target_raw)
D = max(D_pred_raw, D_target_raw)

action_pred = pad_array_to_shape(action_pred_raw, target_len=T, target_dim=D, pad_mode="edge")
action_target = pad_array_to_shape(action_target_raw, target_len=T, target_dim=D, pad_mode="edge")

joint_angles_right_pred, joint_angles_left_pred = split_bimanual_actions(action_pred)
joint_angles_right_target, joint_angles_left_target = split_bimanual_actions(action_target)


# ===================== 数学/几何工具 =====================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0],
                     [0, ca, -sa],
                     [0, sa, ca]], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[ ca, 0, sa],
                     [  0, 1,  0],
                     [-sa, 0, ca]], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    mid = (np.array(base_pos_left, dtype=np.float64) + np.array(base_pos_right, dtype=np.float64)) / 2.0
    pos = mid + np.array([0.0, 0.0, HEAD_HEIGHT_M], dtype=np.float64)
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L/2; y = W/2; z = H/2
    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)
    R = T_world_obj[:3,:3]
    t = T_world_obj[:3,3]
    corners_world = (R @ corners_local.T).T + t[None,:]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a,b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:,0]*axis_len
    y_end = o + R[:,1]*axis_len
    z_end = o + R[:,2]*axis_len
    return o, x_end, y_end, z_end


# ===================== PyBullet FK =====================
def set_joint_states(robot_id: int, angles):
    for i, a in enumerate(angles):
        p.resetJointState(robot_id, i, float(a))

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])
    return points, parent, R_list

def compute_frame_pair(
    robot_id_right_pred, robot_id_left_pred,
    robot_id_right_target, robot_id_left_target,
    f
):
    f_pred = max(0, min(int(f), len(joint_angles_right_pred) - 1))
    f_target = max(0, min(int(f), len(joint_angles_right_target) - 1))

    set_joint_states(robot_id_right_pred, joint_angles_right_pred[f_pred])
    set_joint_states(robot_id_left_pred,  joint_angles_left_pred[f_pred])

    set_joint_states(robot_id_right_target, joint_angles_right_target[f_target])
    set_joint_states(robot_id_left_target,  joint_angles_left_target[f_target])

    pts_r_pred, parent_r_pred, Rr_pred = get_links_world(robot_id_right_pred)
    pts_l_pred, parent_l_pred, Rl_pred = get_links_world(robot_id_left_pred)

    pts_r_target, parent_r_target, Rr_target = get_links_world(robot_id_right_target)
    pts_l_target, parent_l_target, Rl_target = get_links_world(robot_id_left_target)

    return (
        pts_r_pred, parent_r_pred, Rr_pred, pts_l_pred, parent_l_pred, Rl_pred,
        pts_r_target, parent_r_target, Rr_target, pts_l_target, parent_l_target, Rl_target
    )

def skeleton_lines(points, parent):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        pidx = parent[i]
        if pidx >= 0:
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(all_points_list, cams_T_dict):
    all_pts = list(all_points_list)
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3,3][None,:])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs)/2
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span/2
    return center, half


# ===================== 初始化 PyBullet（只一次） =====================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

# pred 机器人位置
base_pos_right_pred = list(base_pos_right)
base_pos_left_pred  = list(base_pos_left)

# target 机器人位置（沿 Y 轴整体平移）
base_pos_right_target = [base_pos_right[0], base_pos_right[1] + COMPARE_OFFSET_Y_M, base_pos_right[2]]
base_pos_left_target  = [base_pos_left[0],  base_pos_left[1]  + COMPARE_OFFSET_Y_M, base_pos_left[2]]

robot_id_right_pred = p.loadURDF(
    urdf_right_path,
    basePosition=base_pos_right_pred,
    baseOrientation=p.getQuaternionFromEuler(base_orn_right),
    useFixedBase=True
)
robot_id_left_pred = p.loadURDF(
    urdf_left_path,
    basePosition=base_pos_left_pred,
    baseOrientation=p.getQuaternionFromEuler(base_orn_left),
    useFixedBase=True
)

robot_id_right_target = p.loadURDF(
    urdf_right_path,
    basePosition=base_pos_right_target,
    baseOrientation=p.getQuaternionFromEuler(base_orn_right),
    useFixedBase=True
)
robot_id_left_target = p.loadURDF(
    urdf_left_path,
    basePosition=base_pos_left_target,
    baseOrientation=p.getQuaternionFromEuler(base_orn_left),
    useFixedBase=True
)

T_world_head = head_cam2world_fixed()


# ===================== Plotly FigureWidget + UI =====================
f0 = 0
(
    pts_r_pred, parent_r_pred, Rr_pred, pts_l_pred, parent_l_pred, Rl_pred,
    pts_r_target, parent_r_target, Rr_target, pts_l_target, parent_l_target, Rl_target
) = compute_frame_pair(
    robot_id_right_pred, robot_id_left_pred,
    robot_id_right_target, robot_id_left_target,
    f0
)

def cams_T_from_frame(pts_r, Rr, pts_l, Rl):
    return {
        "hand_0": cam2world_from_link(pts_l, Rl, wrist_link_idx_left, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "hand_1": cam2world_from_link(pts_r, Rr, wrist_link_idx_right, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":   T_world_head,
    }

# 相机仍然基于 pred 机器人定义
cams_T = cams_T_from_frame(pts_r_pred, Rr_pred, pts_l_pred, Rl_pred)

xr_pred, yr_pred, zr_pred = skeleton_lines(pts_r_pred, parent_r_pred)
xl_pred, yl_pred, zl_pred = skeleton_lines(pts_l_pred, parent_l_pred)

xr_target, yr_target, zr_target = skeleton_lines(pts_r_target, parent_r_target)
xl_target, yl_target, zl_target = skeleton_lines(pts_l_target, parent_l_target)

fig = go.FigureWidget()

# pred 颜色：蓝色系
fig.add_trace(go.Scatter3d(
    x=pts_r_pred[:,0], y=pts_r_pred[:,1], z=pts_r_pred[:,2],
    mode="markers",
    marker=dict(size=4, color="royalblue"),
    name="pred_right_points"
))
fig.add_trace(go.Scatter3d(
    x=xr_pred, y=yr_pred, z=zr_pred,
    mode="lines",
    line=dict(width=5, color="royalblue"),
    name="pred_right_bones"
))
fig.add_trace(go.Scatter3d(
    x=pts_l_pred[:,0], y=pts_l_pred[:,1], z=pts_l_pred[:,2],
    mode="markers",
    marker=dict(size=4, color="deepskyblue"),
    name="pred_left_points"
))
fig.add_trace(go.Scatter3d(
    x=xl_pred, y=yl_pred, z=zl_pred,
    mode="lines",
    line=dict(width=5, color="deepskyblue"),
    name="pred_left_bones"
))

# target 颜色：橙红色系
fig.add_trace(go.Scatter3d(
    x=pts_r_target[:,0], y=pts_r_target[:,1], z=pts_r_target[:,2],
    mode="markers",
    marker=dict(size=4, color="orangered"),
    name="target_right_points"
))
fig.add_trace(go.Scatter3d(
    x=xr_target, y=yr_target, z=zr_target,
    mode="lines",
    line=dict(width=5, color="orangered"),
    name="target_right_bones"
))
fig.add_trace(go.Scatter3d(
    x=pts_l_target[:,0], y=pts_l_target[:,1], z=pts_l_target[:,2],
    mode="markers",
    marker=dict(size=4, color="orange"),
    name="target_left_points"
))
fig.add_trace(go.Scatter3d(
    x=xl_target, y=yl_target, z=zl_target,
    mode="lines",
    line=dict(width=5, color="orange"),
    name="target_left_bones"
))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(
        x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
        mode="lines", line=dict(width=6, color="red"),
        name=f"{prefix}_x", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
        mode="lines", line=dict(width=6, color="green"),
        name=f"{prefix}_y", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
        mode="lines", line=dict(width=6, color="blue"),
        name=f"{prefix}_z", visible=visible, showlegend=False
    ))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x,y,z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z, mode="lines",
        line=dict(width=4, color="purple"),
        name=f"{prefix}_box", visible=visible, showlegend=False
    ))

trace_map = {
    "pred_right_points": [0],
    "pred_right_bones": [1],
    "pred_left_points": [2],
    "pred_left_bones": [3],
    "target_right_points": [4],
    "target_right_bones": [5],
    "target_left_points": [6],
    "target_left_bones": [7],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx+3))

for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx+3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges(
    [pts_r_pred, pts_l_pred, pts_r_target, pts_l_target],
    cams_T
)
fig.update_layout(
    title=(
        f"FK + Cameras (Frame {f0}) | "
        f"pred=blue, target=orange | target Y offset={COMPARE_OFFSET_Y_M:.3f} m"
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=780,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)

# ===================== action 对比图 =====================
action_dim_options = list(range(D))
action_dim_dropdown = widgets.Dropdown(
    options=action_dim_options,
    value=0,
    description="action dim"
)

frame_x = np.arange(T)
action_fig = go.FigureWidget()
d0 = int(action_dim_dropdown.value)

action_fig.add_trace(go.Scatter(
    x=frame_x,
    y=action_pred[:, d0],
    mode="lines",
    name="pred",
    line=dict(width=2, color="royalblue")
))
action_fig.add_trace(go.Scatter(
    x=frame_x,
    y=action_target[:, d0],
    mode="lines",
    name="target",
    line=dict(width=2, color="orangered", dash="dash")
))
action_fig.add_trace(go.Scatter(
    x=[f0],
    y=[action_pred[f0, d0]],
    mode="markers",
    name="pred_cur",
    marker=dict(size=10, symbol="circle", color="royalblue")
))
action_fig.add_trace(go.Scatter(
    x=[f0],
    y=[action_target[f0, d0]],
    mode="markers",
    name="target_cur",
    marker=dict(size=10, symbol="x", color="orangered")
))

action_fig.update_layout(
    title=f"Action Compare (dim={d0}) | pred_len={T_pred_raw}, target_len={T_target_raw}, padded_len={T}",
    height=320,
    margin=dict(l=40, r=20, t=50, b=40),
    xaxis_title="frame",
    yaxis_title="value",
    legend=dict(orientation="h")
)

# --------- UI controls ----------
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_pred = widgets.Checkbox(value=True, description="Show pred")
cb_show_target = widgets.Checkbox(value=True, description="Show target")
cb_show_world = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider, action_dim_dropdown])
ui_row2 = widgets.HBox([cb_show_pred, cb_show_target, cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row3 = widgets.HBox([cam_select, dd_export, btn_export])

# --------- 播放控件（asyncio） ----------
btn_play  = widgets.Button(description="Play ▶", button_style="success")
btn_pause = widgets.Button(description="Pause ⏸", button_style="warning")
btn_step  = widgets.Button(description="Step +1", button_style="")
speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")
ui_row_play = widgets.HBox([btn_play, btn_pause, btn_step, speed, cb_loop])

_play_state = {"running": False, "task": None}

async def _player_loop_async():
    try:
        while _play_state["running"]:
            cur = int(slider.value)
            nxt = cur + 1
            if nxt >= T:
                if cb_loop.value:
                    nxt = 0
                else:
                    _play_state["running"] = False
                    break

            slider.value = nxt
            dt = 1.0 / max(1, int(speed.value))
            await asyncio.sleep(dt)

    except asyncio.CancelledError:
        pass
    except Exception:
        with out_log:
            print("[play][ERROR]")
            traceback.print_exc()
    finally:
        _play_state["running"] = False
        _play_state["task"] = None

def on_play_clicked(_):
    if _play_state["running"]:
        return
    _play_state["running"] = True
    _play_state["task"] = asyncio.create_task(_player_loop_async())
    with out_log:
        print("[play] started")

def on_pause_clicked(_):
    _play_state["running"] = False
    task = _play_state.get("task")
    if task is not None:
        task.cancel()
    with out_log:
        print("[play] paused")

def on_step_clicked(_):
    cur = int(slider.value)
    if cur < T - 1:
        slider.value = cur + 1
    elif cb_loop.value:
        slider.value = 0

btn_play.on_click(on_play_clicked)
btn_pause.on_click(on_pause_clicked)
btn_step.on_click(on_step_clicked)

def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        pred_v = cb_show_pred.value
        target_v = cb_show_target.value

        set_visible(trace_map["pred_right_points"], pred_v)
        set_visible(trace_map["pred_right_bones"], pred_v)
        set_visible(trace_map["pred_left_points"], pred_v)
        set_visible(trace_map["pred_left_bones"], pred_v)

        set_visible(trace_map["target_right_points"], target_v)
        set_visible(trace_map["target_right_bones"], target_v)
        set_visible(trace_map["target_left_points"], target_v)
        set_visible(trace_map["target_left_bones"], target_v)

        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)

def update_action_plot(f):
    f = int(f)
    dim = int(action_dim_dropdown.value)

    with action_fig.batch_update():
        action_fig.data[0].x = frame_x
        action_fig.data[0].y = action_pred[:, dim]

        action_fig.data[1].x = frame_x
        action_fig.data[1].y = action_target[:, dim]

        action_fig.data[2].x = [f]
        action_fig.data[2].y = [action_pred[f, dim]]

        action_fig.data[3].x = [f]
        action_fig.data[3].y = [action_target[f, dim]]

        action_fig.layout.title = (
            f"Action Compare (dim={dim}) | "
            f"pred_len={T_pred_raw}, target_len={T_target_raw}, padded_len={T}"
        )

def update_frame(f):
    f = int(f)
    (
        pts_r_pred, parent_r_pred, Rr_pred, pts_l_pred, parent_l_pred, Rl_pred,
        pts_r_target, parent_r_target, Rr_target, pts_l_target, parent_l_target, Rl_target
    ) = compute_frame_pair(
        robot_id_right_pred, robot_id_left_pred,
        robot_id_right_target, robot_id_left_target,
        f
    )

    xr_pred, yr_pred, zr_pred = skeleton_lines(pts_r_pred, parent_r_pred)
    xl_pred, yl_pred, zl_pred = skeleton_lines(pts_l_pred, parent_l_pred)

    xr_target, yr_target, zr_target = skeleton_lines(pts_r_target, parent_r_target)
    xl_target, yl_target, zl_target = skeleton_lines(pts_l_target, parent_l_target)

    cams_T_new = cams_T_from_frame(pts_r_pred, Rr_pred, pts_l_pred, Rl_pred)
    center, half = scene_ranges(
        [pts_r_pred, pts_l_pred, pts_r_target, pts_l_target],
        cams_T_new
    )

    with fig.batch_update():
        # pred
        fig.data[0].x = pts_r_pred[:,0]; fig.data[0].y = pts_r_pred[:,1]; fig.data[0].z = pts_r_pred[:,2]
        fig.data[1].x = xr_pred;         fig.data[1].y = yr_pred;         fig.data[1].z = zr_pred
        fig.data[2].x = pts_l_pred[:,0]; fig.data[2].y = pts_l_pred[:,1]; fig.data[2].z = pts_l_pred[:,2]
        fig.data[3].x = xl_pred;         fig.data[3].y = yl_pred;         fig.data[3].z = zl_pred

        # target
        fig.data[4].x = pts_r_target[:,0]; fig.data[4].y = pts_r_target[:,1]; fig.data[4].z = pts_r_target[:,2]
        fig.data[5].x = xr_target;         fig.data[5].y = yr_target;         fig.data[5].z = zr_target
        fig.data[6].x = pts_l_target[:,0]; fig.data[6].y = pts_l_target[:,1]; fig.data[6].z = pts_l_target[:,2]
        fig.data[7].x = xl_target;         fig.data[7].y = yl_target;         fig.data[7].z = zl_target

        idx = 8
        idx += 3  # skip world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)
            fig.data[idx+0].x = [o[0], x_end[0]]; fig.data[idx+0].y = [o[1], x_end[1]]; fig.data[idx+0].z = [o[2], x_end[2]]
            fig.data[idx+1].x = [o[0], y_end[0]]; fig.data[idx+1].y = [o[1], y_end[1]]; fig.data[idx+1].z = [o[2], y_end[2]]
            fig.data[idx+2].x = [o[0], z_end[0]]; fig.data[idx+2].y = [o[1], z_end[1]]; fig.data[idx+2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx; fig.data[idx].y = by; fig.data[idx].z = bz
            idx += 1

        fig.layout.title = (
            f"FK + Cameras (Frame {f}) | "
            f"pred=blue, target=orange | target Y offset={COMPARE_OFFSET_Y_M:.3f} m"
        )
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()
    update_action_plot(f)

def on_slider_change(change):
    update_frame(change["new"])

def on_action_dim_change(change):
    update_action_plot(slider.value)

slider.observe(on_slider_change, names="value")
action_dim_dropdown.observe(on_action_dim_change, names="value")

for w in [cb_show_pred, cb_show_target, cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()
update_action_plot(f0)


# ===================== Matplotlib 渲染（导出用） =====================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    (
        pts_r_pred, parent_r_pred, Rr_pred, pts_l_pred, parent_l_pred, Rl_pred,
        pts_r_target, parent_r_target, Rr_target, pts_l_target, parent_l_target, Rl_target
    ) = compute_frame_pair(
        robot_id_right_pred, robot_id_left_pred,
        robot_id_right_target, robot_id_left_target,
        f
    )

    cams_T = cams_T_from_frame(pts_r_pred, Rr_pred, pts_l_pred, Rl_pred)

    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig_m = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig_m.add_subplot(111, projection="3d")

    # pred
    ax.scatter(pts_r_pred[:,0], pts_r_pred[:,1], pts_r_pred[:,2], s=10, c="royalblue")
    for i, pidx in enumerate(parent_r_pred):
        if pidx >= 0:
            ax.plot([pts_r_pred[pidx,0], pts_r_pred[i,0]],
                    [pts_r_pred[pidx,1], pts_r_pred[i,1]],
                    [pts_r_pred[pidx,2], pts_r_pred[i,2]], c="royalblue")

    ax.scatter(pts_l_pred[:,0], pts_l_pred[:,1], pts_l_pred[:,2], s=10, c="deepskyblue")
    for i, pidx in enumerate(parent_l_pred):
        if pidx >= 0:
            ax.plot([pts_l_pred[pidx,0], pts_l_pred[i,0]],
                    [pts_l_pred[pidx,1], pts_l_pred[i,1]],
                    [pts_l_pred[pidx,2], pts_l_pred[i,2]], c="deepskyblue")

    # target
    ax.scatter(pts_r_target[:,0], pts_r_target[:,1], pts_r_target[:,2], s=10, c="orangered")
    for i, pidx in enumerate(parent_r_target):
        if pidx >= 0:
            ax.plot([pts_r_target[pidx,0], pts_r_target[i,0]],
                    [pts_r_target[pidx,1], pts_r_target[i,1]],
                    [pts_r_target[pidx,2], pts_r_target[i,2]], c="orangered")

    ax.scatter(pts_l_target[:,0], pts_l_target[:,1], pts_l_target[:,2], s=10, c="orange")
    for i, pidx in enumerate(parent_l_target):
        if pidx >= 0:
            ax.plot([pts_l_target[pidx,0], pts_l_target[i,0]],
                    [pts_l_target[pidx,1], pts_l_target[i,1]],
                    [pts_l_target[pidx,2], pts_l_target[i,2]], c="orange")

    for Tm in cams_T.values():
        o = Tm[:3,3]; R = Tm[:3,:3]
        ax.quiver(o[0],o[1],o[2], R[0,0],R[1,0],R[2,0], color="r", length=0.08)
        ax.quiver(o[0],o[1],o[2], R[0,1],R[1,1],R[2,1], color="g", length=0.08)
        ax.quiver(o[0],o[1],o[2], R[0,2],R[1,2],R[2,2], color="b", length=0.08)

    all_pts = np.vstack([
        pts_r_pred, pts_l_pred, pts_r_target, pts_l_target,
        np.stack([T[:3,3] for T in cams_T.values()])
    ])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2

    ax.set_xlim(center[0]-half, center[0]+half)
    ax.set_ylim(center[1]-half, center[1]+half)
    ax.set_zlim(center[2]-half, center[2]+half)
    ax.set_box_aspect([1,1,1])
    ax.axis("off")
    ax.set_title(
        f"FK + Cameras (Frame {f}) | pred=blue, target=orange | target Y offset={COMPARE_OFFSET_Y_M:.3f} m"
    )

    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig_m)
    return img, cams_T


# ===================== 保存相机 txt（可选） =====================
def maybe_save_camera_txt(cams_T: dict, frame_idx: int):
    if not SAVE_CAM_TXT:
        return
    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)

    for name in CAMERAS.keys():
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")

    for name, T_wc in cams_T.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), T_wc, fmt="%.9f")


# ===================== 导出（gif/mp4） =====================
def export_frames(export_type: str):
    os.makedirs(EXPORT_DIR, exist_ok=True)

    with out_log:
        clear_output()
        print(f"[export] type={export_type}")
        print("[export] renderer = matplotlib(Agg)")
        print("[export] render_aspect_from =", EXPORT_RENDER_CAM, cam_spec(EXPORT_RENDER_CAM))
        print(f"[export] target Y offset = {COMPARE_OFFSET_Y_M:.3f} m")

    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk.gif")
        frames = []
        for f in range(T):
            img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
            frames.append(img)
            maybe_save_camera_txt(cams_T, f)
            if f % 10 == 0:
                with out_log:
                    print(f"[export] frame {f}/{T-1}")
        imageio.mimsave(out_path, frames, duration=1.0 / FPS)
        with out_log:
            print(f"[export] GIF saved: {out_path}")

    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk.mp4")
        ffmpeg_bin = shutil.which("ffmpeg")
        if ffmpeg_bin is None:
            with out_log:
                print("[export][ERROR] ffmpeg not found. Install: sudo apt-get update && sudo apt-get install -y ffmpeg")
            return

        writer = imageio.get_writer(
            out_path,
            fps=FPS,
            codec="libx264",
            format="FFMPEG",
            ffmpeg_params=["-pix_fmt", "yuv420p"],
        )
        try:
            for f in range(T):
                img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
                writer.append_data(img)
                maybe_save_camera_txt(cams_T, f)
                if f % 10 == 0:
                    with out_log:
                        print(f"[export] frame {f}/{T-1}")
        finally:
            writer.close()

        with out_log:
            print(f"[export] MP4 saved: {out_path}")

    else:
        with out_log:
            print(f"[export] unknown export_type={export_type}")

def on_export_clicked(_):
    typ = dd_export.value
    with out_log:
        clear_output()
    if typ == "none":
        with out_log:
            print("请选择导出格式 gif 或 mp4")
        return
    export_frames(typ)

btn_export.on_click(on_export_clicked)


# ===================== 展示 =====================
with out_log:
    print(f"[info] pred = blue")
    print(f"[info] target = orange")
    print(f"[info] target Y offset = {COMPARE_OFFSET_Y_M:.3f} m")
    print("[info] 这个 offset 是人为加的，仅用于视觉区分，不代表模型误差。")

display(ui_row1, ui_row_play, ui_row2, ui_row3, out_log, fig, action_fig)

Output()

FigureWidget({
    'data': [{'marker': {'color': 'royalblue', 'size': 4},
              'mode': 'markers',
              'name': 'pred_right_points',
              'type': 'scatter3d',
              'uid': '08905e65-e6cc-4e8e-be7b-df3da03e832e',
              'visible': True,
              'x': {'bdata': ('AAAAAAAA0D0AAAAAAABAvgAAAODMO7' ... 'AAwCKc3D8AAAAAvLfgPwAAAOAhpOE/'),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAIFyP2r8AAABgZmbevwAAAEDgzt' ... 'AAgEt41r8AAAAgxnzQvwAAAEDmMdK/'),
                    'dtype': 'f8'},
              'z': {'bdata': ('AAAAAAAA0D0AAAAAAAAwvgAAAIBzZ7' ... 'AAQJ3Ywr8AAAAg6pHIvwAAAAApE8e/'),
                    'dtype': 'f8'}},
             {'line': {'color': 'royalblue', 'width': 5},
              'mode': 'lines',
              'name': 'pred_right_bones',
              'type': 'scatter3d',
              'uid': 'c402a15d-cc2f-4925-b047-d2439c13850a',
              'visible': True,
              'x': array([5.820766091346741e-11, -

FigureWidget({
    'data': [{'line': {'color': 'royalblue', 'width': 2},
              'mode': 'lines',
              'name': 'pred',
              'type': 'scatter',
              'uid': '1517ffc6-ab5d-4003-beb6-e6c7e4921793',
              'x': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'ILAgwCDQIOAg8CEAIRAhICEwIUAg=='),
                    'dtype': 'i2'},
              'y': {'bdata': ('AAAAIDti8L8AAADQ3KXwvwAAAABSUf' ... 'Q7bMG/AAAAhi1Kwb8AAACGLUrBvw=='),
                    'dtype': 'f8'}},
             {'line': {'color': 'orangered', 'dash': 'dash', 'width': 2},
              'mode': 'lines',
              'name': 'target',
              'type': 'scatter',
              'uid': '9505fffc-cee7-4a58-9e23-35bd0f0d5177',
              'x': {'bdata': ('AAABAAIAAwAEAAUABgAHAAgACQAKAA' ... 'ILAgwCDQIOAg8CEAIRAhICEwIUAg=='),
                    'dtype': 'i2'},
              'y': {'bdata': ('AAAAACFO8b8AAAAAIU7xvwAAAAAhTv' ... 'AOTMK/AAAAAA5Mwr8AAAAADkzCvw=='),
                    'dty

In [1]:
json_path_pred = "/liujinxin/code/tram/GR00T-Dreams-main/output/test/106/data_json_test.json"
json_path_target = "/liujinxin/code/tram/GR00T-Dreams-main/output/test/106/data_target_json_test.json"

'/liujinxin/code/tram/GR00T-Dreams-main/output/test/106/data_target_json_test.json'

In [5]:
import json
import numpy as np
from pathlib import Path

left_path  = Path("/liujinxin/dataset/robochallenge/scan_QR_code/data/episode_000188/states/left_states.jsonl")
right_path = Path("/liujinxin/dataset/robochallenge/scan_QR_code/data/episode_000188/states/right_states.jsonl")

def read_jsonl(fp: Path):
    rows = []
    with fp.open("r") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"JSON decode error in {fp} at line {ln}: {e}\nLINE={line[:200]}") from e
    return rows

def get_arm6_and_grip(d: dict):
    # 优先用 qpos；否则 joint_positions
    arm = d.get("qpos", None)
    if arm is None:
        arm = d.get("joint_positions", None)
    if arm is None:
        raise KeyError("No 'qpos' or 'joint_positions' in state dict")
    arm = np.asarray(arm, dtype=np.float64).reshape(-1)
    if arm.shape[0] < 6:
        raise ValueError(f"arm dof < 6, got {arm.shape[0]}")

    arm6 = arm[:6]  # 只取前6维
    grip = float(d.get("gripper", 0.0))
    return arm6, grip

left_rows = read_jsonl(left_path)
right_rows = read_jsonl(right_path)

T = min(len(left_rows), len(right_rows))
if len(left_rows) != len(right_rows):
    print(f"[WARN] left/right length mismatch: left={len(left_rows)}, right={len(right_rows)} -> using T={T}")

actions_14 = np.zeros((T, 14), dtype=np.float64)

for t in range(T):
    l_arm6, l_grip = get_arm6_and_grip(left_rows[t])
    r_arm6, r_grip = get_arm6_and_grip(right_rows[t])

    actions_14[t, 0:6] = l_arm6
    actions_14[t, 6]   = l_grip
    actions_14[t, 7:13]= r_arm6
    actions_14[t, 13]  = r_grip

print("actions_14 shape:", actions_14.shape)
print("actions_14[0]:", actions_14[0])


actions_14 shape: (600, 14)
actions_14[0]: [-6.27809539e-02  4.13422799e-03  3.19225201e-03 -4.20400389e-02
  1.13612771e-01  1.56298243e-02  9.99999975e-05 -9.42673758e-02
  5.65185584e-03  3.10503202e-03  3.11549846e-02  3.41727957e-02
  1.91343233e-01  1.39999995e-03]


In [7]:
print("actions_14 shape:", actions_14.shape)
print("max abs delta:", np.max(np.abs(np.diff(actions_14, axis=0))))
print("first frame:", actions_14[0, :8])
print("last  frame:", actions_14[-1, :8])

actions_14 shape: (600, 14)
max abs delta: 0.09245318174362183
first frame: [-6.27809539e-02  4.13422799e-03  3.19225201e-03 -4.20400389e-02
  1.13612771e-01  1.56298243e-02  9.99999975e-05 -9.42673758e-02]
last  frame: [-1.10682182e-01  9.36742779e-03  2.94803595e-03 -9.60989967e-02
 -2.03397032e-02  4.37321067e-02  9.99999975e-05  9.48953629e-02]


In [9]:
import os, math, shutil, json, asyncio, traceback
import numpy as np
import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 你只需要主要改这里：CAMERAS（每个相机自己的 w/h/fx/fy/cx/cy）
# ============================================================
CAMERAS = {
    "hand_0": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "hand_1": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "head":   dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([[s["fx"], 0.0,     s["cx"]],
                  [0.0,     s["fy"], s["cy"]],
                  [0.0,     0.0,     1.0]], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ===================== 机器人 & 任务参数区（按需改） =====================
# ✅ 单个 ALOHA 双臂 URDF
urdf_bimanual_path = "/liujinxin/code/tram/GR00T-Dreams-main/cosmos-predict2.5/outputs/aloha_new_description/urdf/aloha_new.urdf"

# 机器人 base 放到世界原点（你也可以挪）
base_pos = [0.0, 0.0, 0.0]
base_orn = [0.0, 0.0, 0.0]

# wrist camera 相对 wrist 的安装（cam 坐标：默认朝 +X）
CAM_OFFSET_M = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

# head camera 固定：以 base 为中心 + 高 30cm，俯视 30°
HEAD_HEIGHT_M = 0.30
HEAD_PITCH_DOWN_DEG = -30.0

# 相机盒子尺寸（米）：长x宽x高（局部 X/Y/Z）
CAM_BOX_LWH = (0.06, 0.03, 0.03)

# ========= 你的动作/关节角来源 =========
# 你这里原来是 bimanual 数据：right 7 + left 7+...
# 现在改成读取你自己的 [T,14]（左臂6 左夹1 右臂6 右夹1）
# 你可以用 np.load / torch.load / json 等，下面示例保留 json 读取方式：
# json_path = "/liujinxin/dataset/bimanual/tidy_tools_filtered/1125_bimanual_long/take/pliers/data.json"
# with open(json_path, "r") as f:
#     data = json.load(f)

# T = len(data)

# 这里假设 data 里已经有你要的 14 维关节角（弧度），如果不是请你改解析逻辑
# 下面给一个兼容：优先找 step["action14"]，否则退回 step["joint_angles"] 的某段
# actions_14 = []
# for step in data:
#     if "action14" in step:
#         a = np.array(step["action14"], dtype=np.float64)
#     elif "actions" in step:
#         a = np.array(step["actions"], dtype=np.float64)
#     elif "joint_angles" in step:
#         # ⚠️ 你原数据不是14的话这里会错；你按实际情况改
#         a = np.array(step["joint_angles"][:14], dtype=np.float64)
#     else:
#         raise KeyError("step does not contain action14/actions/joint_angles")
#     assert a.shape[0] == 14, f"expect 14-dim action, got {a.shape}"
#     actions_14.append(a)

# actions_14 = np.stack(actions_14, axis=0)  # (T,14)

# left = [-0.0627809539437294,0.004134227987378836,0.0031922520138323307,-0.04204003885388374,0.11361277103424072,0.015629824250936508]
# right = [-0.09426737576723099,0.00565185584127903,0.0031050320249050856,0.03591719642281532,0.03153875097632408,0.19134323298931122]
# actions_14 = np.array([left+right])


# 导出目录
EXPORT_DIR = "export_fk"
FPS = 10

# 导出渲染采用哪个相机的宽高比（只影响导出视频/动图的“画面比例”）
EXPORT_RENDER_CAM = "head"

# 是否保存相机 txt（intrinsic + 每帧 extrinsic）
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"
# ======================================================================


# ===================== 数学/几何工具 =====================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0],
                     [0, ca, -sa],
                     [0, sa, ca]], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[ ca, 0, sa],
                     [  0, 1,  0],
                     [-sa, 0, ca]], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    """
    约定：cam 的“朝前”是 +X；俯视用绕 Y 旋转（你之前也在用 rot_y）
    """
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    pos = np.array(base_pos, dtype=np.float64) + np.array([0.0, 0.0, HEAD_HEIGHT_M], dtype=np.float64)
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L/2; y = W/2; z = H/2
    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)
    R = T_world_obj[:3,:3]
    t = T_world_obj[:3,3]
    corners_world = (R @ corners_local.T).T + t[None,:]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a,b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:,0]*axis_len
    y_end = o + R[:,1]*axis_len
    z_end = o + R[:,2]*axis_len
    return o, x_end, y_end, z_end


# ===================== PyBullet FK (single bimanual robot) =====================
def list_joints(robot_id: int):
    num = p.getNumJoints(robot_id)
    out = []
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        # info fields: https://pybullet.org/Bullet/BulletFull/pybullet__env_8py_source.html
        jid = int(info[0])
        jname = info[1].decode("utf-8")
        jtype = int(info[2])
        qidx = int(info[3])
        uidx = int(info[4])
        parent = int(info[16])
        link_name = info[12].decode("utf-8")
        out.append(dict(jid=jid, jname=jname, jtype=jtype, qidx=qidx, uidx=uidx, parent=parent, link_name=link_name))
    return out

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])
    return points, parent, R_list

def skeleton_lines(points, parent, mask=None):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        if mask is not None and (not mask[i]):
            continue
        pidx = parent[i]
        if pidx >= 0:
            if mask is not None and (not mask[pidx]):
                continue
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(pts_all, cams_T_dict):
    all_pts = [pts_all]
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3,3][None,:])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs)/2
    span = (maxs - mins).max()
    if span < 1e-6: span = 1.0
    half = span/2
    return center, half


# ===================== ALOHA joint mapping (14-dim -> URDF joints) =====================
# 你这个必须根据 URDF 的关节命名调整一下。
# 跑起来后 out_log 会打印所有 joint name，你据此改 hints。

LEFT_HINTS_ARM   = ["left", "l_", "arm_l", "left_arm"]
RIGHT_HINTS_ARM  = ["right", "r_", "arm_r", "right_arm"]

LEFT_HINTS_GRIP  = ["left", "l_", "gripper", "finger", "hand"]
RIGHT_HINTS_GRIP = ["right", "r_", "gripper", "finger", "hand"]

# 也有人 URDF 用 "arm1/arm2" 这种：你可以把 hints 再加上
# LEFT_HINTS_ARM += ["arm1"]
# RIGHT_HINTS_ARM += ["arm2"]

# ——如果你想完全手动指定 joint 名（最稳），把下面列表填满即可（长度=6+?+6+?）
LEFT_ARM_JOINTS_MANUAL  = None  # e.g. ["left_joint_1", ...]
RIGHT_ARM_JOINTS_MANUAL = None
LEFT_GRIPPER_JOINTS_MANUAL  = None  # e.g. ["left_finger_joint"]
RIGHT_GRIPPER_JOINTS_MANUAL = None

def _match_any(s: str, hints) -> bool:
    s2 = s.lower()
    return any(h in s2 for h in hints)

def build_aloha_joint_groups(robot_id: int):
    joints = list_joints(robot_id)
    revolute_like = []
    for j in joints:
        # jointType: 0=REVOLUTE, 1=PRISMATIC, 2=SPHERICAL, 3=PLANAR, 4=FIXED
        if j["jtype"] in (0, 1):  # revolute/prismatic only
            revolute_like.append(j)

    # 如果手动提供，就直接用手动
    if LEFT_ARM_JOINTS_MANUAL is not None:
        left_arm = LEFT_ARM_JOINTS_MANUAL
    else:
        left_arm = [j["jname"] for j in revolute_like if _match_any(j["jname"], LEFT_HINTS_ARM) and (not _match_any(j["jname"], ["gripper","finger","hand"]))]
        # 常见会包含很多别的 left joints，这里做一次“只取前6个”的策略
        left_arm = left_arm[:6]

    if RIGHT_ARM_JOINTS_MANUAL is not None:
        right_arm = RIGHT_ARM_JOINTS_MANUAL
    else:
        right_arm = [j["jname"] for j in revolute_like if _match_any(j["jname"], RIGHT_HINTS_ARM) and (not _match_any(j["jname"], ["gripper","finger","hand"]))]
        right_arm = right_arm[:6]

    if LEFT_GRIPPER_JOINTS_MANUAL is not None:
        left_grip = LEFT_GRIPPER_JOINTS_MANUAL
    else:
        # 夹爪可能是多个关节，先全拿出来
        left_grip = [j["jname"] for j in revolute_like if _match_any(j["jname"], LEFT_HINTS_GRIP) and _match_any(j["jname"], ["gripper","finger","hand"])]
        # 有些 URDF 夹爪关节名不含 left/right，就会匹配不到：你可以按打印结果手动填
        # left_grip = left_grip[:1]

    if RIGHT_GRIPPER_JOINTS_MANUAL is not None:
        right_grip = RIGHT_GRIPPER_JOINTS_MANUAL
    else:
        right_grip = [j["jname"] for j in revolute_like if _match_any(j["jname"], RIGHT_HINTS_GRIP) and _match_any(j["jname"], ["gripper","finger","hand"])]

    return left_arm, left_grip, right_arm, right_grip

def name_to_joint_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8")
        d[jname] = i
    return d

def set_joint_by_name(robot_id: int, jname: str, value: float, name2idx: dict):
    idx = name2idx.get(jname, None)
    if idx is None:
        return False
    p.resetJointState(robot_id, idx, float(value))
    return True

# ====== 手动指定：把“左臂/右臂”映射到这个URDF里真实存在的6DOF机构 ======
# 你这份 URDF 里明显有 fl_joint1..6 和 fr_joint1..6
LEFT_ARM_JOINTS_MANUAL  = ["fl_joint1","fl_joint2","fl_joint3","fl_joint4","fl_joint5","fl_joint6"]
RIGHT_ARM_JOINTS_MANUAL = ["fr_joint1","fr_joint2","fr_joint3","fr_joint4","fr_joint5","fr_joint6"]

# 这个URDF里没看到 gripper/finger 关节，所以置空（忽略夹爪）
LEFT_GRIPPER_JOINTS_MANUAL  = []
RIGHT_GRIPPER_JOINTS_MANUAL = []


def set_aloha_from_action(robot_id: int, a: np.ndarray, name2idx: dict):
    """
    兼容两种输入：
      - a.shape == (12,) : [left_arm(6), right_arm(6)]
      - a.shape == (14,) : [left_arm(6), left_grip(1), right_arm(6), right_grip(1)]
        （但当前URDF无夹爪关节，grip会被忽略）
    """
    a = np.asarray(a, dtype=np.float64).reshape(-1)

    if a.shape[0] == 12:
        left_arm_vals  = a[0:6]
        right_arm_vals = a[6:12]
        left_grip_val = None
        right_grip_val = None
    elif a.shape[0] == 14:
        left_arm_vals  = a[0:6]
        left_grip_val  = float(a[6])
        right_arm_vals = a[7:13]
        right_grip_val = float(a[13])
    else:
        raise ValueError(f"Expected action dim 12 or 14, got {a.shape[0]}")

    # ---- left arm (6) ----
    if len(LEFT_ARM_JOINTS_MANUAL) != 6:
        raise RuntimeError(f"LEFT_ARM_JOINTS_MANUAL must be 6, got {len(LEFT_ARM_JOINTS_MANUAL)}")
    for k, jn in enumerate(LEFT_ARM_JOINTS_MANUAL):
        idx = name2idx.get(jn, None)
        if idx is None:
            raise RuntimeError(f"Cannot find joint {jn} in URDF")
        p.resetJointState(robot_id, idx, float(left_arm_vals[k]))

    # ---- left gripper (optional) ----
    if left_grip_val is not None and len(LEFT_GRIPPER_JOINTS_MANUAL) > 0:
        for jn in LEFT_GRIPPER_JOINTS_MANUAL:
            idx = name2idx.get(jn, None)
            if idx is not None:
                p.resetJointState(robot_id, idx, float(left_grip_val))

    # ---- right arm (6) ----
    if len(RIGHT_ARM_JOINTS_MANUAL) != 6:
        raise RuntimeError(f"RIGHT_ARM_JOINTS_MANUAL must be 6, got {len(RIGHT_ARM_JOINTS_MANUAL)}")
    for k, jn in enumerate(RIGHT_ARM_JOINTS_MANUAL):
        idx = name2idx.get(jn, None)
        if idx is None:
            raise RuntimeError(f"Cannot find joint {jn} in URDF")
        p.resetJointState(robot_id, idx, float(right_arm_vals[k]))

    # ---- right gripper (optional) ----
    if right_grip_val is not None and len(RIGHT_GRIPPER_JOINTS_MANUAL) > 0:
        for jn in RIGHT_GRIPPER_JOINTS_MANUAL:
            idx = name2idx.get(jn, None)
            if idx is not None:
                p.resetJointState(robot_id, idx, float(right_grip_val))


def compute_frame(robot_id: int, f: int, name2idx: dict):
    """
    单机器人：根据 actions_14[f]（可能是12维或14维）设置关节，然后取所有 link 的世界坐标
    """
    set_aloha_from_action(robot_id, actions_14[f], name2idx)
    pts, parent, R = get_links_world(robot_id)
    return pts, parent, R


# ===================== 自动估计左右腕部 link index（用于相机挂载） =====================
def find_wrist_link_idx(robot_id: int, side: str):
    """
    side: "left" or "right"
    用 link_name/joint_name 猜末端腕部 link
    """
    side = side.lower()
    best = None
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8").lower()
        lname = info[12].decode("utf-8").lower()
        # 尽量匹配 "wrist" / "ee" / "end" 之类
        if side in jname or side in lname:
            if ("wrist" in jname) or ("wrist" in lname) or ("ee" in jname) or ("ee" in lname) or ("end" in jname) or ("end" in lname):
                best = i
    # 如果没找到，就退化为：找 side 相关的最后一个 link
    if best is None:
        for i in range(p.getNumJoints(robot_id)):
            info = p.getJointInfo(robot_id, i)
            jname = info[1].decode("utf-8").lower()
            lname = info[12].decode("utf-8").lower()
            if side in jname or side in lname:
                best = i
    return best


# ===================== 初始化 PyBullet（只一次） =====================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

robot_id = p.loadURDF(
    urdf_bimanual_path,
    basePosition=base_pos,
    baseOrientation=p.getQuaternionFromEuler(base_orn),
    useFixedBase=True
)

# 打印 joint 列表，方便你改 hints
print("=== ALOHA URDF joints ===")
for j in list_joints(robot_id):
    print(f"[{j['jid']:03d}] joint={j['jname']:<40s} link={j['link_name']} type={j['jtype']} parent={j['parent']}")

name2idx = name_to_joint_index(robot_id)

# 手动指定 wrist link（挂相机的位置）
wrist_link_idx_left  = name2idx["fl_joint6"]
wrist_link_idx_right = name2idx["fr_joint6"]


groups = build_aloha_joint_groups(robot_id)
print("\n=== auto joint groups (please verify) ===")
print("left_arm:", groups[0])
print("left_grip:", groups[1])
print("right_arm:", groups[2])
print("right_grip:", groups[3])

# wrist_link_idx_left = find_wrist_link_idx(robot_id, "left")
# wrist_link_idx_right = find_wrist_link_idx(robot_id, "right")
print("\n=== wrist link idx ===")
print("left_wrist_link_idx =", wrist_link_idx_left)
print("right_wrist_link_idx=", wrist_link_idx_right)

if wrist_link_idx_left is None or wrist_link_idx_right is None:
    print("[WARN] wrist link idx not found. You can manually set wrist_link_idx_left/right.")

T_world_head = head_cam2world_fixed()


# ===================== Plotly FigureWidget + UI =====================
f0 = 0
pts, parent, R = compute_frame(robot_id, f0, name2idx)

def cams_T_from_frame(pts, R):
    out = {"head": T_world_head}
    if wrist_link_idx_left is not None:
        out["hand_0"] = cam2world_from_link(pts, R, wrist_link_idx_left, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG)
    else:
        out["hand_0"] = T_world_head
    if wrist_link_idx_right is not None:
        out["hand_1"] = cam2world_from_link(pts, R, wrist_link_idx_right, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG)
    else:
        out["hand_1"] = T_world_head
    return out

cams_T = cams_T_from_frame(pts, R)

# ---- 左右骨架分组（按 link_name/joint_name 关键词） ----
def build_link_mask(robot_id: int, side: str):
    side = side.lower()
    mask = np.zeros((p.getNumJoints(robot_id),), dtype=bool)
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8").lower()
        lname = info[12].decode("utf-8").lower()
        if side in jname or side in lname:
            mask[i] = True
    return mask

mask_left = build_link_mask(robot_id, "left")
mask_right = build_link_mask(robot_id, "right")
# 如果 mask 太少（比如 URDF 没 left/right 关键字），就直接全显示在两边（不分了）
if mask_left.sum() < 3 or mask_right.sum() < 3:
    mask_left = None
    mask_right = None

xr, yr, zr = skeleton_lines(pts, parent, mask=mask_right)
xl, yl, zl = skeleton_lines(pts, parent, mask=mask_left)

fig = go.FigureWidget()
# 右
fig.add_trace(go.Scatter3d(x=pts[:,0], y=pts[:,1], z=pts[:,2], mode="markers",
                           marker=dict(size=3), name="all_points"))
fig.add_trace(go.Scatter3d(x=xr, y=yr, z=zr, mode="lines", line=dict(width=4), name="right_bones"))
fig.add_trace(go.Scatter3d(x=xl, y=yl, z=zl, mode="lines", line=dict(width=4), name="left_bones"))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
                               mode="lines", line=dict(width=6, color="red"),
                               name=f"{prefix}_x", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
                               mode="lines", line=dict(width=6, color="green"),
                               name=f"{prefix}_y", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
                               mode="lines", line=dict(width=6, color="blue"),
                               name=f"{prefix}_z", visible=visible, showlegend=False))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x,y,z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(width=4, color="purple"),
                               name=f"{prefix}_box", visible=visible, showlegend=False))

trace_map = {
    "all_points": [0],
    "right_bones": [1],
    "left_bones": [2],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

# world axes
base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx+3))

# camera axes + boxes
for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx+3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges(pts, cams_T)
fig.update_layout(
    title=f"ALOHA bimanual FK + Cameras (Frame {f0}) — drag/zoom/pan",
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)

# --------- UI controls ----------
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_points = widgets.Checkbox(value=True, description="All points")
cb_show_right  = widgets.Checkbox(value=True, description="Right bones")
cb_show_left   = widgets.Checkbox(value=True, description="Left bones")
cb_show_world  = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider])
ui_row2 = widgets.HBox([cb_show_points, cb_show_right, cb_show_left, cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row3 = widgets.HBox([cam_select, dd_export, btn_export])

# --------- 播放控件（稳定版：ipywidgets.Play） ----------
play = widgets.Play(
    interval=int(1000 / FPS),  # ms
    value=int(slider.value),
    min=0,
    max=T-1,
    step=1,
    description="Play",
    disabled=False,
)
# 让 play.value <-> slider.value 联动
widgets.jslink((play, "value"), (slider, "value"))

speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")

def _on_speed_change(change):
    play.interval = int(1000 / max(1, int(change["new"])))

speed.observe(_on_speed_change, names="value")

# Loop：到末尾时回到0
def _on_play_value(change):
    if not cb_loop.value:
        return
    if int(change["new"]) >= T - 1:
        play.value = 0

play.observe(_on_play_value, names="value")

ui_row_play = widgets.HBox([play, speed, cb_loop])


def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        set_visible(trace_map["all_points"], cb_show_points.value)
        set_visible(trace_map["right_bones"], cb_show_right.value)
        set_visible(trace_map["left_bones"], cb_show_left.value)
        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)

def update_frame(f):
    f = int(f)

    # # DEBUG：确认 slider 变化会触发这里
    # if f % 30 == 0:
    #     with out_log:
    #         print(f"[update_frame] f={f}")

    # ✅ 注意：这里不再传 groups
    pts, parent, R = compute_frame(robot_id, f, name2idx)

    if f % 30 == 0:
        with out_log:
            print("  left_wrist xyz =", pts[wrist_link_idx_left])
            print("  right_wrist xyz=", pts[wrist_link_idx_right])

    cams_T_new = cams_T_from_frame(pts, R)
    center, half = scene_ranges(pts, cams_T_new)

    xr, yr, zr = skeleton_lines(pts, parent, mask=mask_right)
    xl, yl, zl = skeleton_lines(pts, parent, mask=mask_left)

    with fig.batch_update():
        fig.data[0].x = pts[:,0]; fig.data[0].y = pts[:,1]; fig.data[0].z = pts[:,2]
        fig.data[1].x = xr;       fig.data[1].y = yr;       fig.data[1].z = zr
        fig.data[2].x = xl;       fig.data[2].y = yl;       fig.data[2].z = zl

        idx = 3
        idx += 3  # skip world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)
            fig.data[idx+0].x = [o[0], x_end[0]]; fig.data[idx+0].y = [o[1], x_end[1]]; fig.data[idx+0].z = [o[2], x_end[2]]
            fig.data[idx+1].x = [o[0], y_end[0]]; fig.data[idx+1].y = [o[1], y_end[1]]; fig.data[idx+1].z = [o[2], y_end[2]]
            fig.data[idx+2].x = [o[0], z_end[0]]; fig.data[idx+2].y = [o[1], z_end[1]]; fig.data[idx+2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx; fig.data[idx].y = by; fig.data[idx].z = bz
            idx += 1

        fig.layout.title = f"ALOHA FK + Cameras (Frame {f}) — drag/zoom/pan"
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()

def on_slider_change(change):
    update_frame(change["new"])

slider.observe(on_slider_change, names="value")

for w in [cb_show_points, cb_show_right, cb_show_left, cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()


# ===================== Matplotlib 渲染（导出用） =====================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    # ✅ 注意：这里也不再传 groups
    pts, parent, R = compute_frame(robot_id, f, name2idx)
    cams_T = cams_T_from_frame(pts, R)

    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=10)

    xr, yr, zr = skeleton_lines(pts, parent, mask=mask_right)
    ax.plot(xr.astype(float), yr.astype(float), zr.astype(float), linewidth=2)

    xl, yl, zl = skeleton_lines(pts, parent, mask=mask_left)
    ax.plot(xl.astype(float), yl.astype(float), zl.astype(float), linewidth=2)

    for Tm in cams_T.values():
        o = Tm[:3,3]; RR = Tm[:3,:3]
        ax.quiver(o[0],o[1],o[2], RR[0,0],RR[1,0],RR[2,0], length=0.08)
        ax.quiver(o[0],o[1],o[2], RR[0,1],RR[1,1],RR[2,1], length=0.08)
        ax.quiver(o[0],o[1],o[2], RR[0,2],RR[1,2],RR[2,2], length=0.08)

    all_pts = np.vstack([pts, np.stack([T[:3,3] for T in cams_T.values()])])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2
    span = (maxs - mins).max()
    if span < 1e-6: span = 1.0
    half = span / 2

    ax.set_xlim(center[0]-half, center[0]+half)
    ax.set_ylim(center[1]-half, center[1]+half)
    ax.set_zlim(center[2]-half, center[2]+half)
    ax.set_box_aspect([1,1,1])
    ax.axis("off")
    ax.set_title(f"ALOHA FK + Cameras (Frame {f})")

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig)
    return img, cams_T


# ===================== 展示 =====================
display(ui_row1, ui_row_play, ui_row2, ui_row3, out_log, fig)

b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
footprint=== ALOHA URDF joints ===
[000] joint=base_joint                               link=base_link type=4 parent=-1
[001] joint=inertial_joint                           link=inertial_link type=4 parent=0
[002] joint=right_wheel                              link=right_wheel_link type=0 parent=0
[003] joint=left_wheel                               link=left_wheel_link type=0 parent=0
[004] joint=fl_castor_wheel                          link=fl_castor_link type=0 parent=0
[005] joint=fl_wheel                                 link=fl_wheel_link type=0 parent=4
[006] joint=fr_castor_wheel                          link=fr_castor_link type=0 parent=0
[007] joint=fr_wheel                                 link=fr_wheel_link type=0 parent=6
[008] joint

Output()

FigureWidget({
    'data': [{'marker': {'size': 3},
              'mode': 'markers',
              'name': 'all_points',
              'type': 'scatter3d',
              'uid': '966cbec7-ac2a-405b-a614-2762355653a1',
              'visible': True,
              'x': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AgJb3UvwAAAGB3IdC/AAAAYHch0L8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAGCPws' ... 'DA35bTvwAAAEDjltO/AAAAQOOW078='),
                    'dtype': 'f8'},
              'z': {'bdata': ('AAAAQDMzwz8AAABAMzPDPwAAAMByaL' ... 'DAqqvxPwAAAMDvE/E/AAAAoO8T8T8='),
                    'dtype': 'f8'}},
             {'line': {'width': 4},
              'mode': 'lines',
              'name': 'right_bones',
              'type': 'scatter3d',
              'uid': 'cd798042-941b-45ce-854c-67cd9cda545b',
              'visible': True,
              'x': array([0.0, 0.0, None, 0.0, 0.0, None, 0.0, 0.0, None, 0.0, 0.1899999976158142,
       

NameError: name 'out_path' is not defined

In [4]:
a = [1,2,3,4]
b = [0,3]
print(a[b])

TypeError: list indices must be integers or slices, not list

In [ ]:
## Aloha 双臂 双轨迹显示
import json
import os, math, shutil
import numpy as np
from pathlib import Path

import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 相机参数
# ============================================================
CAMERAS = {
    "hand_0": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "hand_1": dict(w=640,  h=480,  fx=450.0, fy=450.0, cx=None, cy=None),
    "head":   dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([[s["fx"], 0.0,     s["cx"]],
                  [0.0,     s["fy"], s["cy"]],
                  [0.0,     0.0,     1.0]], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ===================== 机器人 & 任务参数区（按需改） =====================
urdf_bimanual_path = "/liujinxin/code/tram/GR00T-Dreams-main/cosmos-predict2.5/outputs/aloha_new_description/urdf/aloha_new.urdf"

# 第一台机器人 base（episode_000188）
base_pos = [0.0, 0.0, 0.0]
base_orn = [0.0, 0.0, 0.0]

# ============================================================
# ✅ 两台机器人对比时，第二台机器人整体沿 Y 轴平移的距离（单位：米）
# 这是人为视觉错开，不是轨迹误差
# 如果你想完全重合看，把它改成 0.0
# ============================================================
ROBOT_COMPARE_OFFSET_Y_M = 0.

CAM_OFFSET_M = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

HEAD_HEIGHT_M = 0.30
HEAD_PITCH_DOWN_DEG = -30.0

CAM_BOX_LWH = (0.06, 0.03, 0.03)

EXPORT_DIR = "export_fk"
FPS = 10
EXPORT_RENDER_CAM = "head"
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"


# ===================== 两个 episode 的路径 =====================
ep1_left_path  = Path("/liujinxin/dataset/robochallenge/turn_on_faucet/data/episode_000000/states/left_states.jsonl")
ep1_right_path = Path("/liujinxin/dataset/robochallenge/turn_on_faucet/data/episode_000000/states/right_states.jsonl")

ep2_left_path  = Path("/liujinxin/dataset/robochallenge/turn_on_faucet/data/episode_000000/states/left_states.jsonl")
ep2_right_path = Path("/liujinxin/dataset/robochallenge/turn_on_faucet/data/episode_000000/states/lower_right_states.jsonl")

# ep1_left_path  = Path("/liujinxin/code/tram/GR00T-Dreams-main/output/test/163/left_states_target.jsonl")
# ep1_right_path = Path("/liujinxin/code/tram/GR00T-Dreams-main/output/test/163/right_states_target.jsonl")

# ep2_left_path  = Path("/liujinxin/code/tram/GR00T-Dreams-main/output/test/163/left_states_pred.jsonl")
# ep2_right_path = Path("/liujinxin/code/tram/GR00T-Dreams-main/output/test/163/right_states_pred.jsonl")


# ep1_left_path  = Path("/liujinxin/dataset/robochallenge/robochallenge_scan_QR_code_filtered_predict_videos_actions/0/states/left_states.jsonl")
# ep1_right_path = Path("/liujinxin/dataset/robochallenge/robochallenge_scan_QR_code_filtered_predict_videos_actions/0/states/right_states.jsonl")

# ep2_left_path = ep1_left_path
# ep2_right_path = ep1_right_path


# ===================== 读取 episode 工具 =====================
def read_jsonl(fp: Path):
    rows = []
    with fp.open("r") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"JSON decode error in {fp} at line {ln}: {e}\nLINE={line[:200]}") from e
    return rows

# def get_arm6_and_grip(d: dict):
#     arm = d.get("qpos", None)
#     if arm is None:
#         arm = d.get("joint_positions", None)
#     if arm is None:
#         raise KeyError("No 'qpos' or 'joint_positions' in state dict")

#     arm = np.asarray(arm, dtype=np.float64).reshape(-1)
#     if arm.shape[0] < 6:
#         raise ValueError(f"arm dof < 6, got {arm.shape[0]}")

#     arm6 = arm[:6]
#     grip = float(d.get("gripper", 0.0))
#     return arm6, grip

def get_arm6_and_grip(d: dict):
    arm = d.get("joint_positions", None)   # ✅ 强制用这个

    if arm is None:
        raise KeyError("No 'joint_positions' in state dict")

    arm = np.asarray(arm, dtype=np.float64).reshape(-1)
    arm6 = arm[:6]

    grip = float(d.get("gripper", 0.0))
    return arm6, grip

def build_actions_14(left_path: Path, right_path: Path):
    left_rows = read_jsonl(left_path)
    right_rows = read_jsonl(right_path)

    T_local = min(len(left_rows), len(right_rows))
    if len(left_rows) != len(right_rows):
        print(f"[WARN] {left_path.parent.parent.name}: left/right length mismatch: "
              f"left={len(left_rows)}, right={len(right_rows)} -> using T={T_local}")

    actions_14 = np.zeros((T_local, 14), dtype=np.float64)
    for t in range(T_local):
        l_arm6, l_grip = get_arm6_and_grip(left_rows[t])
        r_arm6, r_grip = get_arm6_and_grip(right_rows[t])

        actions_14[t, 0:6]   = l_arm6
        actions_14[t, 6]     = l_grip
        actions_14[t, 7:13]  = r_arm6
        actions_14[t, 13]    = r_grip

    return actions_14

def pad_actions_edge(arr, target_len):
    arr = np.asarray(arr, dtype=np.float64)
    T0, D0 = arr.shape
    if T0 == target_len:
        return arr
    if T0 > target_len:
        return arr[:target_len]
    if T0 == 0:
        return np.zeros((target_len, D0), dtype=np.float64)
    pad_rows = np.repeat(arr[-1:, :], target_len - T0, axis=0)
    return np.concatenate([arr, pad_rows], axis=0)


# ===================== 读取两个 episode =====================
actions_14_ep1_raw = build_actions_14(ep1_left_path, ep1_right_path)
actions_14_ep2_raw = build_actions_14(ep2_left_path, ep2_right_path)


# actions_14_ep1_raw = build_actions_14(ep1_left_path, ep1_right_path)

# # clone 一份作为 ep2
# actions_14_ep2_raw = actions_14_ep1_raw.copy()

# # 对每一帧的“右手臂最后一个关节”手动 +10 度
# # 你的 14 维定义是：
# # [left_arm(6), left_grip(1), right_arm(6), right_grip(1)]
# # 所以 right_arm 是索引 [7:13]，最后一个关节是全局索引 12
# RIGHT_ARM_LAST_JOINT_IDX = 10
# DELTA_DEG = -10.0
# DELTA_RAD = np.deg2rad(DELTA_DEG)
# actions_14_ep1_raw[:, RIGHT_ARM_LAST_JOINT_IDX] += DELTA_RAD
# actions_14_ep1_raw[:, 11] += np.deg2rad(10)
# # actions_14_ep2_raw[:, 9] += np.deg2rad(10)


# RIGHT_ARM_LAST_JOINT_IDX = 10
# DELTA_DEG = -10.0
# DELTA_RAD = np.deg2rad(DELTA_DEG)
# print(f"actions_14_ep2_raw[0, 10]: {actions_14_ep2_raw[0, RIGHT_ARM_LAST_JOINT_IDX]}")
# actions_14_ep2_raw[:, RIGHT_ARM_LAST_JOINT_IDX] += DELTA_RAD
# print(f"actions_14_ep2_raw[0, 10]: {actions_14_ep2_raw[0, RIGHT_ARM_LAST_JOINT_IDX]}")
# print(f"actions_14_ep2_raw[0, 11]: {actions_14_ep2_raw[0, 11]}")
# actions_14_ep2_raw[:, 11] += np.deg2rad(10)
# print(f"actions_14_ep2_raw[0, 11]: {actions_14_ep2_raw[0, 11]}")

# actions_14_ep3_raw = []
# first_time = True

# for index_actions_14_ep2_raw in range(20, len(actions_14_ep2_raw)):
#     if actions_14_ep2_raw[index_actions_14_ep2_raw, -1]<0.03 and first_time is True:
#         edict_state = actions_14_ep2_raw[index_actions_14_ep2_raw].copy()
#         edict_state[9] += np.deg2rad(5)
#         actions_14_ep3_raw.append(edict_state.copy())
#         edict_state[9] += np.deg2rad(5)
#         actions_14_ep3_raw.append(edict_state.copy())
        
#         edict_state[11] += np.deg2rad(-5)
#         actions_14_ep3_raw.append(edict_state.copy())
#         edict_state[11] += np.deg2rad(-5)
#         actions_14_ep3_raw.append(edict_state.copy())
#         edict_state[11] += np.deg2rad(-5)
#         actions_14_ep3_raw.append(edict_state.copy())
        
#         first_time = False
    
#     elif actions_14_ep2_raw[index_actions_14_ep2_raw, -1]<0.03 and first_time is False:
#         actions_14_ep2_raw[index_actions_14_ep2_raw, 9] += np.deg2rad(10)
#         actions_14_ep2_raw[index_actions_14_ep2_raw, 11] += np.deg2rad(-15)
#         actions_14_ep3_raw.append(actions_14_ep2_raw[index_actions_14_ep2_raw].copy())
#     else:
#         actions_14_ep3_raw.append(actions_14_ep2_raw[index_actions_14_ep2_raw].copy())

# actions_14_ep3_raw = np.array(actions_14_ep3_raw)
# print(f"actions_14_ep3_raw: {actions_14_ep3_raw.shape}")


# print(f"[INFO] ep2 cloned from ep1, and joint[{RIGHT_ARM_LAST_JOINT_IDX}] += {DELTA_DEG} deg ({DELTA_RAD:.6f} rad)")


T1 = actions_14_ep1_raw.shape[0]
T2 = actions_14_ep2_raw.shape[0]
T = max(T1, T2)

actions_14_ep1 = pad_actions_edge(actions_14_ep1_raw, T)
actions_14_ep2 = pad_actions_edge(actions_14_ep2_raw, T)

print("actions_14_ep1 shape:", actions_14_ep1.shape)
print("actions_14_ep2 shape:", actions_14_ep2.shape)
print(f"[INFO] padded length T = {T}")


# ===================== 数学/几何工具 =====================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[1, 0, 0],
                     [0, ca, -sa],
                     [0, sa, ca]], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([[ ca, 0, sa],
                     [  0, 1,  0],
                     [-sa, 0, ca]], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    pos = np.array(base_pos, dtype=np.float64) + np.array([0.0, 0.0, HEAD_HEIGHT_M], dtype=np.float64)
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L/2; y = W/2; z = H/2
    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)
    R = T_world_obj[:3,:3]
    t = T_world_obj[:3,3]
    corners_world = (R @ corners_local.T).T + t[None,:]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a,b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:,0]*axis_len
    y_end = o + R[:,1]*axis_len
    z_end = o + R[:,2]*axis_len
    return o, x_end, y_end, z_end


# ===================== PyBullet FK =====================
def list_joints(robot_id: int):
    num = p.getNumJoints(robot_id)
    out = []
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        jid = int(info[0])
        jname = info[1].decode("utf-8")
        jtype = int(info[2])
        qidx = int(info[3])
        uidx = int(info[4])
        parent = int(info[16])
        link_name = info[12].decode("utf-8")
        out.append(dict(jid=jid, jname=jname, jtype=jtype, qidx=qidx, uidx=uidx, parent=parent, link_name=link_name))
    return out

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])
    return points, parent, R_list

def skeleton_lines(points, parent, mask=None):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        if mask is not None and (not mask[i]):
            continue
        pidx = parent[i]
        if pidx >= 0:
            if mask is not None and (not mask[pidx]):
                continue
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(points_list, cams_T_dict):
    all_pts = list(points_list)
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3,3][None,:])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs)/2
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span/2
    return center, half


# ===================== ALOHA joint mapping =====================
LEFT_ARM_JOINTS_MANUAL  = ["fl_joint1","fl_joint2","fl_joint3","fl_joint4","fl_joint5","fl_joint6"]
RIGHT_ARM_JOINTS_MANUAL = ["fr_joint1","fr_joint2","fr_joint3","fr_joint4","fr_joint5","fr_joint6"]

# 当前 URDF 没看到 gripper/finger joint，就置空
LEFT_GRIPPER_JOINTS_MANUAL  = []
RIGHT_GRIPPER_JOINTS_MANUAL = []

def name_to_joint_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8")
        d[jname] = i
    return d

def set_aloha_from_action(robot_id: int, a: np.ndarray, name2idx: dict):
    """
    a.shape == (14,) : [left_arm(6), left_grip(1), right_arm(6), right_grip(1)]
    当前 gripper 没对应 joint，因此 grip 会忽略
    """
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    if a.shape[0] != 14:
        raise ValueError(f"Expected action dim 14, got {a.shape[0]}")

    left_arm_vals   = a[0:6]
    left_grip_val   = float(a[6])
    right_arm_vals  = a[7:13]
    right_grip_val  = float(a[13])

    if len(LEFT_ARM_JOINTS_MANUAL) != 6:
        raise RuntimeError(f"LEFT_ARM_JOINTS_MANUAL must be 6, got {len(LEFT_ARM_JOINTS_MANUAL)}")
    if len(RIGHT_ARM_JOINTS_MANUAL) != 6:
        raise RuntimeError(f"RIGHT_ARM_JOINTS_MANUAL must be 6, got {len(RIGHT_ARM_JOINTS_MANUAL)}")

    for k, jn in enumerate(LEFT_ARM_JOINTS_MANUAL):
        idx = name2idx.get(jn, None)
        if idx is None:
            raise RuntimeError(f"Cannot find joint {jn} in URDF")
        p.resetJointState(robot_id, idx, float(left_arm_vals[k]))

    for jn in LEFT_GRIPPER_JOINTS_MANUAL:
        idx = name2idx.get(jn, None)
        if idx is not None:
            p.resetJointState(robot_id, idx, float(left_grip_val))

    for k, jn in enumerate(RIGHT_ARM_JOINTS_MANUAL):
        idx = name2idx.get(jn, None)
        if idx is None:
            raise RuntimeError(f"Cannot find joint {jn} in URDF")
        p.resetJointState(robot_id, idx, float(right_arm_vals[k]))

    for jn in RIGHT_GRIPPER_JOINTS_MANUAL:
        idx = name2idx.get(jn, None)
        if idx is not None:
            p.resetJointState(robot_id, idx, float(right_grip_val))

def compute_frame(robot_id: int, f: int, name2idx: dict, actions_14: np.ndarray):
    set_aloha_from_action(robot_id, actions_14[f], name2idx)
    pts, parent, R = get_links_world(robot_id)
    return pts, parent, R


# ===================== 初始化 PyBullet =====================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

# 第一台机器人：episode_000188
robot_id_ep1 = p.loadURDF(
    urdf_bimanual_path,
    basePosition=base_pos,
    baseOrientation=p.getQuaternionFromEuler(base_orn),
    useFixedBase=True
)

# 第二台机器人：episode_000189，沿 Y 轴错开
base_pos_ep2 = [base_pos[0], base_pos[1] + ROBOT_COMPARE_OFFSET_Y_M, base_pos[2]]
robot_id_ep2 = p.loadURDF(
    urdf_bimanual_path,
    basePosition=base_pos_ep2,
    baseOrientation=p.getQuaternionFromEuler(base_orn),
    useFixedBase=True
)

print("=== ALOHA URDF joints ===")
for j in list_joints(robot_id_ep1):
    print(f"[{j['jid']:03d}] joint={j['jname']:<40s} link={j['link_name']} type={j['jtype']} parent={j['parent']}")

name2idx_ep1 = name_to_joint_index(robot_id_ep1)
name2idx_ep2 = name_to_joint_index(robot_id_ep2)

# wrist link
wrist_link_idx_left  = name2idx_ep1["fl_joint6"]
wrist_link_idx_right = name2idx_ep1["fr_joint6"]

print("\n=== wrist link idx ===")
print("left_wrist_link_idx =", wrist_link_idx_left)
print("right_wrist_link_idx=", wrist_link_idx_right)
print(f"[INFO] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
print("[INFO] 这是第二个机器人沿 Y 轴的人为视觉偏移，不是动作误差。")

T_world_head = head_cam2world_fixed()


# ===================== 左右 mask =====================
def build_link_mask(robot_id: int, side_prefix: str):
    """
    这份 URDF 直接按 fl_/fr_ 分
    """
    mask = np.zeros((p.getNumJoints(robot_id),), dtype=bool)
    side_prefix = side_prefix.lower()
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8").lower()
        lname = info[12].decode("utf-8").lower()
        if jname.startswith(side_prefix) or lname.startswith(side_prefix) or side_prefix in jname or side_prefix in lname:
            mask[i] = True
    return mask

mask_left_ep1  = build_link_mask(robot_id_ep1, "fl_")
mask_right_ep1 = build_link_mask(robot_id_ep1, "fr_")
mask_left_ep2  = build_link_mask(robot_id_ep2, "fl_")
mask_right_ep2 = build_link_mask(robot_id_ep2, "fr_")


# ===================== Plotly FigureWidget + UI =====================
f0 = 0
pts1, parent1, R1 = compute_frame(robot_id_ep1, f0, name2idx_ep1, actions_14_ep1)
pts2, parent2, R2 = compute_frame(robot_id_ep2, f0, name2idx_ep2, actions_14_ep2)

def cams_T_from_frame(pts, R):
    return {
        "hand_0": cam2world_from_link(pts, R, wrist_link_idx_left, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "hand_1": cam2world_from_link(pts, R, wrist_link_idx_right, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":   T_world_head,
    }

# 相机先挂第一台机器人上
cams_T = cams_T_from_frame(pts1, R1)

xr1, yr1, zr1 = skeleton_lines(pts1, parent1, mask=mask_right_ep1)
xl1, yl1, zl1 = skeleton_lines(pts1, parent1, mask=mask_left_ep1)

xr2, yr2, zr2 = skeleton_lines(pts2, parent2, mask=mask_right_ep2)
xl2, yl2, zl2 = skeleton_lines(pts2, parent2, mask=mask_left_ep2)

fig = go.FigureWidget()

# episode_000188：蓝色系
fig.add_trace(go.Scatter3d(
    x=pts1[:,0], y=pts1[:,1], z=pts1[:,2],
    mode="markers",
    marker=dict(size=3, color="royalblue"),
    name="ep188_points"
))
fig.add_trace(go.Scatter3d(
    x=xr1, y=yr1, z=zr1,
    mode="lines",
    line=dict(width=5, color="royalblue"),
    name="ep188_right_bones"
))
fig.add_trace(go.Scatter3d(
    x=xl1, y=yl1, z=zl1,
    mode="lines",
    line=dict(width=5, color="deepskyblue"),
    name="ep188_left_bones"
))

# episode_000189：橙色系
fig.add_trace(go.Scatter3d(
    x=pts2[:,0], y=pts2[:,1], z=pts2[:,2],
    mode="markers",
    marker=dict(size=3, color="orangered"),
    name="ep189_points"
))
fig.add_trace(go.Scatter3d(
    x=xr2, y=yr2, z=zr2,
    mode="lines",
    line=dict(width=5, color="orangered"),
    name="ep189_right_bones"
))
fig.add_trace(go.Scatter3d(
    x=xl2, y=yl2, z=zl2,
    mode="lines",
    line=dict(width=5, color="orange"),
    name="ep189_left_bones"
))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
                               mode="lines", line=dict(width=6, color="red"),
                               name=f"{prefix}_x", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
                               mode="lines", line=dict(width=6, color="green"),
                               name=f"{prefix}_y", visible=visible, showlegend=False))
    fig.add_trace(go.Scatter3d(x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
                               mode="lines", line=dict(width=6, color="blue"),
                               name=f"{prefix}_z", visible=visible, showlegend=False))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x,y,z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines",
                               line=dict(width=4, color="purple"),
                               name=f"{prefix}_box", visible=visible, showlegend=False))

trace_map = {
    "ep188_points": [0],
    "ep188_right_bones": [1],
    "ep188_left_bones": [2],
    "ep189_points": [3],
    "ep189_right_bones": [4],
    "ep189_left_bones": [5],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx+3))

for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx+3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges([pts1, pts2], cams_T)
fig.update_layout(
    title=(
        f"ALOHA FK Compare (Frame {f0}) | "
        f"ep188=blue, ep189=orange | "
        f"ep189 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m"
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)


# --------- UI controls ----------
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_ep188_points = widgets.Checkbox(value=True, description="ep188 points")
cb_show_ep188_right  = widgets.Checkbox(value=True, description="ep188 right")
cb_show_ep188_left   = widgets.Checkbox(value=True, description="ep188 left")

cb_show_ep189_points = widgets.Checkbox(value=True, description="ep189 points")
cb_show_ep189_right  = widgets.Checkbox(value=True, description="ep189 right")
cb_show_ep189_left   = widgets.Checkbox(value=True, description="ep189 left")

cb_show_world = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider])
ui_row2 = widgets.HBox([
    cb_show_ep188_points, cb_show_ep188_right, cb_show_ep188_left,
    cb_show_ep189_points, cb_show_ep189_right, cb_show_ep189_left
])
ui_row3 = widgets.HBox([cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row4 = widgets.HBox([cam_select, dd_export, btn_export])

# --------- 播放控件 ----------
play = widgets.Play(
    interval=int(1000 / FPS),
    value=int(slider.value),
    min=0,
    max=T-1,
    step=1,
    description="Play",
    disabled=False,
)
widgets.jslink((play, "value"), (slider, "value"))

speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")

def _on_speed_change(change):
    play.interval = int(1000 / max(1, int(change["new"])))
speed.observe(_on_speed_change, names="value")

def _on_play_value(change):
    if not cb_loop.value:
        return
    if int(change["new"]) >= T - 1:
        play.value = 0
play.observe(_on_play_value, names="value")

ui_row_play = widgets.HBox([play, speed, cb_loop])


def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        set_visible(trace_map["ep188_points"], cb_show_ep188_points.value)
        set_visible(trace_map["ep188_right_bones"], cb_show_ep188_right.value)
        set_visible(trace_map["ep188_left_bones"], cb_show_ep188_left.value)

        set_visible(trace_map["ep189_points"], cb_show_ep189_points.value)
        set_visible(trace_map["ep189_right_bones"], cb_show_ep189_right.value)
        set_visible(trace_map["ep189_left_bones"], cb_show_ep189_left.value)

        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)

def update_frame(f):
    f = int(f)

    pts1, parent1, R1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_14_ep1)
    pts2, parent2, R2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_14_ep2)

    cams_T_new = cams_T_from_frame(pts1, R1)
    center, half = scene_ranges([pts1, pts2], cams_T_new)

    xr1, yr1, zr1 = skeleton_lines(pts1, parent1, mask=mask_right_ep1)
    xl1, yl1, zl1 = skeleton_lines(pts1, parent1, mask=mask_left_ep1)

    xr2, yr2, zr2 = skeleton_lines(pts2, parent2, mask=mask_right_ep2)
    xl2, yl2, zl2 = skeleton_lines(pts2, parent2, mask=mask_left_ep2)

    with fig.batch_update():
        fig.data[0].x = pts1[:,0]; fig.data[0].y = pts1[:,1]; fig.data[0].z = pts1[:,2]
        fig.data[1].x = xr1;       fig.data[1].y = yr1;       fig.data[1].z = zr1
        fig.data[2].x = xl1;       fig.data[2].y = yl1;       fig.data[2].z = zl1

        fig.data[3].x = pts2[:,0]; fig.data[3].y = pts2[:,1]; fig.data[3].z = pts2[:,2]
        fig.data[4].x = xr2;       fig.data[4].y = yr2;       fig.data[4].z = zr2
        fig.data[5].x = xl2;       fig.data[5].y = yl2;       fig.data[5].z = zl2

        idx = 6
        idx += 3  # skip world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)
            fig.data[idx+0].x = [o[0], x_end[0]]; fig.data[idx+0].y = [o[1], x_end[1]]; fig.data[idx+0].z = [o[2], x_end[2]]
            fig.data[idx+1].x = [o[0], y_end[0]]; fig.data[idx+1].y = [o[1], y_end[1]]; fig.data[idx+1].z = [o[2], y_end[2]]
            fig.data[idx+2].x = [o[0], z_end[0]]; fig.data[idx+2].y = [o[1], z_end[1]]; fig.data[idx+2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx; fig.data[idx].y = by; fig.data[idx].z = bz
            idx += 1

        fig.layout.title = (
            f"ALOHA FK Compare (Frame {f}) | "
            f"ep188=blue, ep189=orange | "
            f"ep189 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m"
        )
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()

def on_slider_change(change):
    update_frame(change["new"])

slider.observe(on_slider_change, names="value")

for w in [
    cb_show_ep188_points, cb_show_ep188_right, cb_show_ep188_left,
    cb_show_ep189_points, cb_show_ep189_right, cb_show_ep189_left,
    cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select
]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()


# ===================== Matplotlib 渲染（导出用） =====================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    pts1, parent1, R1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_14_ep1)
    pts2, parent2, R2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_14_ep2)

    cams_T = cams_T_from_frame(pts1, R1)

    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig_m = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig_m.add_subplot(111, projection="3d")

    # ep188 蓝色
    ax.scatter(pts1[:,0], pts1[:,1], pts1[:,2], s=10, c="royalblue")
    xr1, yr1, zr1 = skeleton_lines(pts1, parent1, mask=mask_right_ep1)
    xl1, yl1, zl1 = skeleton_lines(pts1, parent1, mask=mask_left_ep1)
    ax.plot(xr1.astype(float), yr1.astype(float), zr1.astype(float), linewidth=2, c="royalblue")
    ax.plot(xl1.astype(float), yl1.astype(float), zl1.astype(float), linewidth=2, c="deepskyblue")

    # ep189 橙色
    ax.scatter(pts2[:,0], pts2[:,1], pts2[:,2], s=10, c="orangered")
    xr2, yr2, zr2 = skeleton_lines(pts2, parent2, mask=mask_right_ep2)
    xl2, yl2, zl2 = skeleton_lines(pts2, parent2, mask=mask_left_ep2)
    ax.plot(xr2.astype(float), yr2.astype(float), zr2.astype(float), linewidth=2, c="orangered")
    ax.plot(xl2.astype(float), yl2.astype(float), zl2.astype(float), linewidth=2, c="orange")

    for Tm in cams_T.values():
        o = Tm[:3,3]
        RR = Tm[:3,:3]
        ax.quiver(o[0],o[1],o[2], RR[0,0],RR[1,0],RR[2,0], length=0.08, color="r")
        ax.quiver(o[0],o[1],o[2], RR[0,1],RR[1,1],RR[2,1], length=0.08, color="g")
        ax.quiver(o[0],o[1],o[2], RR[0,2],RR[1,2],RR[2,2], length=0.08, color="b")

    all_pts = np.vstack([pts1, pts2, np.stack([T[:3,3] for T in cams_T.values()])])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2

    ax.set_xlim(center[0]-half, center[0]+half)
    ax.set_ylim(center[1]-half, center[1]+half)
    ax.set_zlim(center[2]-half, center[2]+half)
    ax.set_box_aspect([1,1,1])
    ax.axis("off")
    ax.set_title(
        f"ALOHA FK Compare (Frame {f}) | ep188=blue, ep189=orange | "
        f"ep189 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m"
    )

    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig_m)
    return img, cams_T


# ===================== 保存相机 txt（可选） =====================
def maybe_save_camera_txt(cams_T: dict, frame_idx: int):
    if not SAVE_CAM_TXT:
        return
    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)

    for name in CAMERAS.keys():
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")

    for name, T_wc in cams_T.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), T_wc, fmt="%.9f")


# ===================== 导出（gif/mp4） =====================
def export_frames(export_type: str):
    os.makedirs(EXPORT_DIR, exist_ok=True)

    with out_log:
        clear_output()
        print(f"[export] type={export_type}")
        print("[export] renderer = matplotlib(Agg)")
        print("[export] render_aspect_from =", EXPORT_RENDER_CAM, cam_spec(EXPORT_RENDER_CAM))
        print(f"[export] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
        print("[export] 这是第二个机器人的人为视觉偏移，不是轨迹误差。")

    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.gif")
        frames = []
        for f in range(T):
            img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
            frames.append(img)
            maybe_save_camera_txt(cams_T, f)
            if f % 10 == 0:
                with out_log:
                    print(f"[export] frame {f}/{T-1}")
        imageio.mimsave(out_path, frames, duration=1.0 / FPS)
        with out_log:
            print(f"[export] GIF saved: {out_path}")

    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.mp4")
        ffmpeg_bin = shutil.which("ffmpeg")
        if ffmpeg_bin is None:
            with out_log:
                print("[export][ERROR] ffmpeg not found. Install: sudo apt-get update && sudo apt-get install -y ffmpeg")
            return

        writer = imageio.get_writer(
            out_path,
            fps=FPS,
            codec="libx264",
            format="FFMPEG",
            ffmpeg_params=["-pix_fmt", "yuv420p"],
        )
        try:
            for f in range(T):
                img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
                writer.append_data(img)
                maybe_save_camera_txt(cams_T, f)
                if f % 10 == 0:
                    with out_log:
                        print(f"[export] frame {f}/{T-1}")
        finally:
            writer.close()

        with out_log:
            print(f"[export] MP4 saved: {out_path}")

    else:
        with out_log:
            print(f"[export] unknown export_type={export_type}")

def on_export_clicked(_):
    typ = dd_export.value
    with out_log:
        clear_output()
    if typ == "none":
        with out_log:
            print("请选择导出格式 gif 或 mp4")
        return
    export_frames(typ)

btn_export.on_click(on_export_clicked)


# ===================== 展示 =====================
with out_log:
    print("[info] ep188 = blue")
    print("[info] ep189 = orange")
    print(f"[info] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
    print("[info] 这个变量表示第二个机器人沿 Y 轴的人为视觉错开距离，不是动作误差。")
    print(f"[info] ep188 original length = {T1}")
    print(f"[info] ep189 original length = {T2}")
    print(f"[info] padded display length = {T}")

display(ui_row1, ui_row_play, ui_row2, ui_row3, ui_row4, out_log, fig)

actions_14_ep1 shape: (800, 14)
actions_14_ep2 shape: (800, 14)
[INFO] padded length T = 800
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
footprint=== ALOHA URDF joints ===b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
base_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
footprintb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, loc

Output()

FigureWidget({
    'data': [{'marker': {'color': 'royalblue', 'size': 3},
              'mode': 'markers',
              'name': 'ep188_points',
              'type': 'scatter3d',
              'uid': '2b096b78-a223-4e87-a77d-810219581631',
              'visible': True,
              'x': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AgJb3UvwAAAGB3IdC/AAAAYHch0L8='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAGCPws' ... 'DA35bTvwAAAEDjltO/AAAAQOOW078='),
                    'dtype': 'f8'},
              'z': {'bdata': ('AAAAQDMzwz8AAABAMzPDPwAAAMByaL' ... 'DAqqvxPwAAAMDvE/E/AAAAoO8T8T8='),
                    'dtype': 'f8'}},
             {'line': {'color': 'royalblue', 'width': 5},
              'mode': 'lines',
              'name': 'ep188_right_bones',
              'type': 'scatter3d',
              'uid': '805eb0af-ec80-4278-808f-08404f439475',
              'visible': True,
              'x': array([0.1895499974489212, 0.167741

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

...

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

[BAD] i=5
  type(x) = <class 'numpy.ndarray'>
  arr.shape = (800, 14)
  arr = [[ 0.00591352 -0.00294804 -0.01140838 ...  0.21946867  0.01428664
  -0.0025    ]
 [ 0.00591352 -0.00294804 -0.01140838 ...  0.21946867  0.01428664
  -0.0025    ]
 [ 0.00591352 -0.00294804 -0.01140838 ...  0.21946867  0.01428664
  -0.0025    ]
 ...
 [ 0.00591352 -0.00294804 -0.01140838 ...  0.14475602  0.01320511
  -0.0023    ]
 [ 0.00591352 -0.00294804 -0.01140838 ...  0.14475602  0.01320511
  -0.0023    ]
 [ 0.00591352 -0.00294804 -0.01140838 ...  0.14475602  0.01320511
  -0.0023    ]]


In [ ]:
## UR5e 单臂 双轨迹显示
# -*- coding: utf-8 -*-

import json
import os
import math
import shutil
import numpy as np
from pathlib import Path

import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 相机参数
# ============================================================
CAMERAS = {
    "wrist": dict(w=640,  h=480,  fx=450.0,  fy=450.0,  cx=None, cy=None),
    "head":  dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([
        [s["fx"], 0.0,     s["cx"]],
        [0.0,     s["fy"], s["cy"]],
        [0.0,     0.0,     1.0]
    ], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ============================================================
# ✅ 机器人 & 路径参数区（按需改）
# ============================================================

# URDF 路径
urdf_path = "/liujinxin/code/tram/cosmos-predict2.5/outputs/ur5e/pybullet_ur5_robotiq-robotflow/urdf/ur5.urdf"

# 第一条轨迹
ep1_path = Path("/liujinxin/code/tram/lingbot-va/ur5e_states_target.jsonl")

# 第二条轨迹
ep2_path = Path("/liujinxin/code/tram/lingbot-va/ur5e_states.jsonl")

# 机器人 base
base_pos = [0.0, 0.0, 0.0]
base_orn = [0.0, 0.0, 0.0]

# 第二台机器人沿 Y 轴偏移，仅用于视觉比较
ROBOT_COMPARE_OFFSET_Y_M = 0.0

# UR5/UR5e 常见 6 关节名
UR5E_JOINTS = [
    "shoulder_pan_joint",
    "shoulder_lift_joint",
    "elbow_joint",
    "wrist_1_joint",
    "wrist_2_joint",
    "wrist_3_joint",
]

# 末端候选 link / joint 名
EE_CANDIDATE_NAMES = [
    "tool0",
    "ee_link",
    "tcp_link",
    "wrist_3_link",
    "wrist_3_joint",
    "ee_fixed_joint",
]

CAM_OFFSET_M = np.array([0.06, 0.0, 0.04], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

HEAD_HEIGHT_M = 0.60
HEAD_PITCH_DOWN_DEG = -30.0

CAM_BOX_LWH = (0.06, 0.03, 0.03)

EXPORT_DIR = "export_fk_ur5e_compare"
FPS = 10
EXPORT_RENDER_CAM = "head"
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"


# ============================================================
# ✅ 读取 jsonl
# ============================================================
def read_jsonl(fp: Path):
    rows = []
    with fp.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(
                    f"JSON decode error in {fp} at line {ln}: {e}\nLINE={line[:200]}"
                ) from e
    return rows

def get_arm6(d: dict):
    arm = d.get("joint_positions", None)
    if arm is None:
        arm = d.get("qpos", None)
    if arm is None:
        raise KeyError("No 'joint_positions' or 'qpos' in state dict")

    arm = np.asarray(arm, dtype=np.float64).reshape(-1)
    if arm.shape[0] < 6:
        raise ValueError(f"Expected at least 6 joints, got {arm.shape[0]}")
    return arm[:6]

def build_actions_6(state_path: Path):
    rows = read_jsonl(state_path)
    actions_6 = np.zeros((len(rows), 6), dtype=np.float64)
    for t, d in enumerate(rows):
        actions_6[t] = get_arm6(d)
    return actions_6

def pad_actions_edge(arr, target_len):
    arr = np.asarray(arr, dtype=np.float64)
    T0, D0 = arr.shape
    if T0 == target_len:
        return arr
    if T0 > target_len:
        return arr[:target_len]
    if T0 == 0:
        return np.zeros((target_len, D0), dtype=np.float64)
    pad_rows = np.repeat(arr[-1:, :], target_len - T0, axis=0)
    return np.concatenate([arr, pad_rows], axis=0)


# ============================================================
# ✅ 读取两条轨迹
# ============================================================
actions_6_ep1_raw = build_actions_6(ep1_path)
actions_6_ep2_raw = build_actions_6(ep2_path)

# import numpy as np

# path = "/liujinxin/code/tram/lingbot-va/received_data/20260402_112856_935_job_test/actions_second.npy"

# actions_6_ep1_raw = np.load(path)[:, :6]
# print(f"actions_6_ep1_raw: {actions_6_ep1_raw.shape}")
# actions_6_ep2_raw = actions_6_ep1_raw.copy()

T1 = actions_6_ep1_raw.shape[0]
T2 = actions_6_ep2_raw.shape[0]
T = max(T1, T2)

actions_6_ep1 = pad_actions_edge(actions_6_ep1_raw, T)
actions_6_ep2 = pad_actions_edge(actions_6_ep2_raw, T)

print("actions_6_ep1 shape:", actions_6_ep1.shape)
print("actions_6_ep2 shape:", actions_6_ep2.shape)
print(f"[INFO] padded length T = {T}")


# ============================================================
# ✅ 数学/几何工具
# ============================================================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([
        [1, 0, 0],
        [0, ca, -sa],
        [0, sa, ca]
    ], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([
        [ ca, 0, sa],
        [  0, 1,  0],
        [-sa, 0, ca]
    ], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    pos = np.array(base_pos, dtype=np.float64) + np.array([0.0, 0.0, HEAD_HEIGHT_M], dtype=np.float64)
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, pos)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L / 2
    y = W / 2
    z = H / 2

    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)

    R = T_world_obj[:3, :3]
    t = T_world_obj[:3, 3]
    corners_world = (R @ corners_local.T).T + t[None, :]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a, b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:, 0] * axis_len
    y_end = o + R[:, 1] * axis_len
    z_end = o + R[:, 2] * axis_len
    return o, x_end, y_end, z_end


# ============================================================
# ✅ PyBullet FK
# ============================================================
def list_joints(robot_id: int):
    num = p.getNumJoints(robot_id)
    out = []
    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        jid = int(info[0])
        jname = info[1].decode("utf-8")
        jtype = int(info[2])
        qidx = int(info[3])
        uidx = int(info[4])
        parent = int(info[16])
        link_name = info[12].decode("utf-8")
        out.append(dict(
            jid=jid,
            jname=jname,
            jtype=jtype,
            qidx=qidx,
            uidx=uidx,
            parent=parent,
            link_name=link_name
        ))
    return out

def name_to_joint_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jname = info[1].decode("utf-8")
        d[jname] = i
    return d

def name_to_link_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        link_name = info[12].decode("utf-8")
        joint_name = info[1].decode("utf-8")
        d[link_name] = i
        d[joint_name] = i
    return d

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)

    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])

    return points, parent, R_list

def skeleton_lines(points, parent, mask=None):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        if mask is not None and (not mask[i]):
            continue
        pidx = parent[i]
        if pidx >= 0:
            if mask is not None and (not mask[pidx]):
                continue
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(points_list, cams_T_dict):
    all_pts = list(points_list)
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3, 3][None, :])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2.0
    return center, half

def find_ee_link_index(robot_id: int):
    name2link = name_to_link_index(robot_id)
    for name in EE_CANDIDATE_NAMES:
        if name in name2link:
            return name2link[name], name
    raise RuntimeError("Cannot find end-effector link. Tried: " + ", ".join(EE_CANDIDATE_NAMES))

def set_ur5e_from_action(robot_id: int, a: np.ndarray, name2idx: dict):
    a = np.asarray(a, dtype=np.float64).reshape(-1)
    if a.shape[0] != 6:
        raise ValueError(f"Expected action dim 6, got {a.shape[0]}")

    for k, jn in enumerate(UR5E_JOINTS):
        idx = name2idx.get(jn, None)
        if idx is None:
            raise RuntimeError(f"Cannot find joint {jn} in URDF")
        p.resetJointState(robot_id, idx, float(a[k]))

def compute_frame(robot_id: int, f: int, name2idx: dict, actions_6: np.ndarray):
    set_ur5e_from_action(robot_id, actions_6[f], name2idx)
    pts, parent, R = get_links_world(robot_id)
    return pts, parent, R


# ============================================================
# ✅ 初始化 PyBullet
# ============================================================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

# 第一台机器人：轨迹 1
robot_id_ep1 = p.loadURDF(
    urdf_path,
    basePosition=base_pos,
    baseOrientation=p.getQuaternionFromEuler(base_orn),
    useFixedBase=True
)

# 第二台机器人：轨迹 2，沿 Y 轴平移
base_pos_ep2 = [base_pos[0], base_pos[1] + ROBOT_COMPARE_OFFSET_Y_M, base_pos[2]]
robot_id_ep2 = p.loadURDF(
    urdf_path,
    basePosition=base_pos_ep2,
    baseOrientation=p.getQuaternionFromEuler(base_orn),
    useFixedBase=True
)

print("=== UR5e URDF joints ===")
for j in list_joints(robot_id_ep1):
    print(
        f"[{j['jid']:03d}] joint={j['jname']:<35s} "
        f"link={j['link_name']:<35s} type={j['jtype']} parent={j['parent']}"
    )

name2idx_ep1 = name_to_joint_index(robot_id_ep1)
name2idx_ep2 = name_to_joint_index(robot_id_ep2)

ee_link_idx_ep1, ee_link_name_ep1 = find_ee_link_index(robot_id_ep1)
ee_link_idx_ep2, ee_link_name_ep2 = find_ee_link_index(robot_id_ep2)

print("\n=== ee link ===")
print("ep1 ee_link_idx =", ee_link_idx_ep1, "ee_link_name =", ee_link_name_ep1)
print("ep2 ee_link_idx =", ee_link_idx_ep2, "ee_link_name =", ee_link_name_ep2)
print(f"[INFO] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
print("[INFO] 这是第二个机器人的人为视觉偏移，不是动作误差。")

T_world_head = head_cam2world_fixed()


# ============================================================
# ✅ 全链路 mask（单臂）
# ============================================================
mask_ep1 = np.ones((p.getNumJoints(robot_id_ep1),), dtype=bool)
mask_ep2 = np.ones((p.getNumJoints(robot_id_ep2),), dtype=bool)


# ============================================================
# ✅ 初始帧
# ============================================================
f0 = 0
pts1, parent1, R1 = compute_frame(robot_id_ep1, f0, name2idx_ep1, actions_6_ep1)
pts2, parent2, R2 = compute_frame(robot_id_ep2, f0, name2idx_ep2, actions_6_ep2)

def cams_T_from_frame(pts, R, ee_link_idx):
    return {
        "wrist": cam2world_from_link(pts, R, ee_link_idx, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":  T_world_head,
    }

# 相机先挂第一台机器人上
cams_T = cams_T_from_frame(pts1, R1, ee_link_idx_ep1)

x1, y1, z1 = skeleton_lines(pts1, parent1, mask=mask_ep1)
x2, y2, z2 = skeleton_lines(pts2, parent2, mask=mask_ep2)


# ============================================================
# ✅ Plotly FigureWidget + UI
# ============================================================
fig = go.FigureWidget()

# ep1：蓝色
fig.add_trace(go.Scatter3d(
    x=pts1[:, 0], y=pts1[:, 1], z=pts1[:, 2],
    mode="markers",
    marker=dict(size=3, color="royalblue"),
    name="ep1_points"
))
fig.add_trace(go.Scatter3d(
    x=x1, y=y1, z=z1,
    mode="lines",
    line=dict(width=5, color="royalblue"),
    name="ep1_bones"
))

# ep2：橙色
fig.add_trace(go.Scatter3d(
    x=pts2[:, 0], y=pts2[:, 1], z=pts2[:, 2],
    mode="markers",
    marker=dict(size=3, color="orangered"),
    name="ep2_points"
))
fig.add_trace(go.Scatter3d(
    x=x2, y=y2, z=z2,
    mode="lines",
    line=dict(width=5, color="orangered"),
    name="ep2_bones"
))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(
        x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
        mode="lines", line=dict(width=6, color="red"),
        name=f"{prefix}_x", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
        mode="lines", line=dict(width=6, color="green"),
        name=f"{prefix}_y", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
        mode="lines", line=dict(width=6, color="blue"),
        name=f"{prefix}_z", visible=visible, showlegend=False
    ))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x, y, z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines",
        line=dict(width=4, color="purple"),
        name=f"{prefix}_box", visible=visible, showlegend=False
    ))

trace_map = {
    "ep1_points": [0],
    "ep1_bones": [1],
    "ep2_points": [2],
    "ep2_bones": [3],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx + 3))

for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx + 3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges([pts1, pts2], cams_T)
fig.update_layout(
    title=(
        f"UR5e FK Compare (Frame {f0}) | "
        f"ep1=blue, ep2=orange | "
        f"ep2 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m | "
        f"ee={ee_link_name_ep1}"
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)


# ============================================================
# ✅ UI controls
# ============================================================
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_ep1_points = widgets.Checkbox(value=True, description="ep1 points")
cb_show_ep1_bones  = widgets.Checkbox(value=True, description="ep1 bones")
cb_show_ep2_points = widgets.Checkbox(value=True, description="ep2 points")
cb_show_ep2_bones  = widgets.Checkbox(value=True, description="ep2 bones")

cb_show_world = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider])
ui_row2 = widgets.HBox([
    cb_show_ep1_points, cb_show_ep1_bones,
    cb_show_ep2_points, cb_show_ep2_bones
])
ui_row3 = widgets.HBox([cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row4 = widgets.HBox([cam_select, dd_export, btn_export])

# --------- 播放控件 ----------
play = widgets.Play(
    interval=int(1000 / FPS),
    value=int(slider.value),
    min=0,
    max=T-1,
    step=1,
    description="Play",
    disabled=False,
)
widgets.jslink((play, "value"), (slider, "value"))

speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")

def _on_speed_change(change):
    play.interval = int(1000 / max(1, int(change["new"])))

speed.observe(_on_speed_change, names="value")

def _on_play_value(change):
    if not cb_loop.value:
        return
    if int(change["new"]) >= T - 1:
        play.value = 0

play.observe(_on_play_value, names="value")

ui_row_play = widgets.HBox([play, speed, cb_loop])


# ============================================================
# ✅ 显隐控制
# ============================================================
def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        set_visible(trace_map["ep1_points"], cb_show_ep1_points.value)
        set_visible(trace_map["ep1_bones"],  cb_show_ep1_bones.value)

        set_visible(trace_map["ep2_points"], cb_show_ep2_points.value)
        set_visible(trace_map["ep2_bones"],  cb_show_ep2_bones.value)

        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)


# ============================================================
# ✅ 更新帧
# ============================================================
def update_frame(f):
    f = int(f)

    pts1, parent1, R1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_6_ep1)
    pts2, parent2, R2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_6_ep2)

    cams_T_new = cams_T_from_frame(pts1, R1, ee_link_idx_ep1)
    center, half = scene_ranges([pts1, pts2], cams_T_new)

    x1, y1, z1 = skeleton_lines(pts1, parent1, mask=mask_ep1)
    x2, y2, z2 = skeleton_lines(pts2, parent2, mask=mask_ep2)

    with fig.batch_update():
        fig.data[0].x = pts1[:, 0]
        fig.data[0].y = pts1[:, 1]
        fig.data[0].z = pts1[:, 2]

        fig.data[1].x = x1
        fig.data[1].y = y1
        fig.data[1].z = z1

        fig.data[2].x = pts2[:, 0]
        fig.data[2].y = pts2[:, 1]
        fig.data[2].z = pts2[:, 2]

        fig.data[3].x = x2
        fig.data[3].y = y2
        fig.data[3].z = z2

        idx = 4
        idx += 3  # skip world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)

            fig.data[idx + 0].x = [o[0], x_end[0]]
            fig.data[idx + 0].y = [o[1], x_end[1]]
            fig.data[idx + 0].z = [o[2], x_end[2]]

            fig.data[idx + 1].x = [o[0], y_end[0]]
            fig.data[idx + 1].y = [o[1], y_end[1]]
            fig.data[idx + 1].z = [o[2], y_end[2]]

            fig.data[idx + 2].x = [o[0], z_end[0]]
            fig.data[idx + 2].y = [o[1], z_end[1]]
            fig.data[idx + 2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx
            fig.data[idx].y = by
            fig.data[idx].z = bz
            idx += 1

        fig.layout.title = (
            f"UR5e FK Compare (Frame {f}) | "
            f"ep1=blue, ep2=orange | "
            f"ep2 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m | "
            f"ee={ee_link_name_ep1}"
        )
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()

def on_slider_change(change):
    update_frame(change["new"])

slider.observe(on_slider_change, names="value")

for w in [
    cb_show_ep1_points, cb_show_ep1_bones,
    cb_show_ep2_points, cb_show_ep2_bones,
    cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select
]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()


# ============================================================
# ✅ Matplotlib 渲染（导出用）
# ============================================================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    pts1, parent1, R1 = compute_frame(robot_id_ep1, f, name2idx_ep1, actions_6_ep1)
    pts2, parent2, R2 = compute_frame(robot_id_ep2, f, name2idx_ep2, actions_6_ep2)

    cams_T = cams_T_from_frame(pts1, R1, ee_link_idx_ep1)

    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig_m = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig_m.add_subplot(111, projection="3d")

    # ep1 蓝色
    ax.scatter(pts1[:, 0], pts1[:, 1], pts1[:, 2], s=10, c="royalblue")
    x1, y1, z1 = skeleton_lines(pts1, parent1, mask=mask_ep1)
    ax.plot(x1.astype(float), y1.astype(float), z1.astype(float), linewidth=2, c="royalblue")

    # ep2 橙色
    ax.scatter(pts2[:, 0], pts2[:, 1], pts2[:, 2], s=10, c="orangered")
    x2, y2, z2 = skeleton_lines(pts2, parent2, mask=mask_ep2)
    ax.plot(x2.astype(float), y2.astype(float), z2.astype(float), linewidth=2, c="orangered")

    for Tm in cams_T.values():
        o = Tm[:3, 3]
        RR = Tm[:3, :3]
        ax.quiver(o[0], o[1], o[2], RR[0, 0], RR[1, 0], RR[2, 0], length=0.08, color="r")
        ax.quiver(o[0], o[1], o[2], RR[0, 1], RR[1, 1], RR[2, 1], length=0.08, color="g")
        ax.quiver(o[0], o[1], o[2], RR[0, 2], RR[1, 2], RR[2, 2], length=0.08, color="b")

    all_pts = np.vstack([pts1, pts2, np.stack([T[:3, 3] for T in cams_T.values()])])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2.0

    ax.set_xlim(center[0] - half, center[0] + half)
    ax.set_ylim(center[1] - half, center[1] + half)
    ax.set_zlim(center[2] - half, center[2] + half)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")
    ax.set_title(
        f"UR5e FK Compare (Frame {f}) | ep1=blue, ep2=orange | "
        f"ep2 Y offset={ROBOT_COMPARE_OFFSET_Y_M:.3f} m | ee={ee_link_name_ep1}"
    )

    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig_m)
    return img, cams_T


# ============================================================
# ✅ 保存相机 txt
# ============================================================
def maybe_save_camera_txt(cams_T: dict, frame_idx: int):
    if not SAVE_CAM_TXT:
        return

    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)

    for name in CAMERAS.keys():
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")

    for name, T_wc in cams_T.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), T_wc, fmt="%.9f")


# ============================================================
# ✅ 导出 gif/mp4
# ============================================================
def export_frames(export_type: str):
    os.makedirs(EXPORT_DIR, exist_ok=True)

    with out_log:
        clear_output()
        print(f"[export] type={export_type}")
        print("[export] renderer = matplotlib(Agg)")
        print("[export] render_aspect_from =", EXPORT_RENDER_CAM, cam_spec(EXPORT_RENDER_CAM))
        print(f"[export] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
        print("[export] 这是第二个机器人的人为视觉偏移，不是轨迹误差。")

    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.gif")
        frames = []
        for f in range(T):
            img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
            frames.append(img)
            maybe_save_camera_txt(cams_T, f)
            if f % 10 == 0:
                with out_log:
                    print(f"[export] frame {f}/{T-1}")
        imageio.mimsave(out_path, frames, duration=1.0 / FPS)
        with out_log:
            print(f"[export] GIF saved: {out_path}")

    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.mp4")
        ffmpeg_bin = shutil.which("ffmpeg")
        if ffmpeg_bin is None:
            with out_log:
                print("[export][ERROR] ffmpeg not found. Install: sudo apt-get update && sudo apt-get install -y ffmpeg")
            return

        writer = imageio.get_writer(
            out_path,
            fps=FPS,
            codec="libx264",
            format="FFMPEG",
            ffmpeg_params=["-pix_fmt", "yuv420p"],
        )
        try:
            for f in range(T):
                img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
                writer.append_data(img)
                maybe_save_camera_txt(cams_T, f)
                if f % 10 == 0:
                    with out_log:
                        print(f"[export] frame {f}/{T-1}")
        finally:
            writer.close()

        with out_log:
            print(f"[export] MP4 saved: {out_path}")

    else:
        with out_log:
            print(f"[export] unknown export_type={export_type}")

def on_export_clicked(_):
    typ = dd_export.value
    with out_log:
        clear_output()
    if typ == "none":
        with out_log:
            print("请选择导出格式 gif 或 mp4")
        return
    export_frames(typ)

btn_export.on_click(on_export_clicked)


# ============================================================
# ✅ 展示
# ============================================================
with out_log:
    print("[info] ep1 = blue")
    print("[info] ep2 = orange")
    print(f"[info] ROBOT_COMPARE_OFFSET_Y_M = {ROBOT_COMPARE_OFFSET_Y_M:.3f} m")
    print("[info] 这个变量表示第二个机器人沿 Y 轴的人为视觉错开距离，不是动作误差。")
    print(f"[info] ep1 original length = {T1}")
    print(f"[info] ep2 original length = {T2}")
    print(f"[info] padded display length = {T}")
    print(f"[info] ee_link_name = {ee_link_name_ep1}")

display(ui_row1, ui_row_play, ui_row2, ui_row3, ui_row4, out_log, fig)

pybullet build time: Jan 29 2025 23:16:28


actions_6_ep1_raw: (32, 6)
actions_6_ep1 shape: (32, 6)
actions_6_ep2 shape: (32, 6)
[INFO] padded length T = 32
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frame=== UR5e URDF joints ===
[000] joint=shoulder_pan_joint                  link=shoulder_link                       type=0 parent=-1
[001] joint=shoulder_lift_joint                 link=upper_arm_link                      type=0 parent=0
[002] joint=elbow_joint                         link=forearm_link                        type=0 parent=1
[003] joint=wrist_1_joint                       link=wrist_1_link                        type=0 parent=2
[004] joint=wrist_2_joint                       link=wrist_2_link                        type=0 parent=3
[005] joint=wrist_3_joint                       link=wrist_3_link                        type=0 parent=4
[006] joint=ee_fixed_joint                      link=ee_li

Output()

FigureWidget({
    'data': [{'marker': {'color': 'royalblue', 'size': 3},
              'mode': 'markers',
              'name': 'ep1_points',
              'type': 'scatter3d',
              'uid': '6f60cd3b-2a87-4209-874f-c6a73f06baee',
              'visible': True,
              'x': {'bdata': 'AAAAAAAAAAAAAACAAEGbvwAAAIAciMW/AAAAoPGq4b8AAAAgNEDivwAAAGCEOOW/AAAAoFUy5b8=',
                    'dtype': 'f8'},
              'y': {'bdata': 'AAAAAAAAAAAAAAAARA3BPwAAAIC3jJG/AAAAALMFuL8AAAAg355lvwAAAOCxsJW/AAAAIFGJlb8=',
                    'dtype': 'f8'},
              'z': {'bdata': 'AAAAwB/Ttj8AAACAH9O2PwAAAMCkrt4/AAAAQJAf4D8AAABAkB/gPwAAAEBQGOA/AAAAgEjs2j8=',
                    'dtype': 'f8'}},
             {'line': {'color': 'royalblue', 'width': 5},
              'mode': 'lines',
              'name': 'ep1_bones',
              'type': 'scatter3d',
              'uid': '2834371c-95c1-4816-b05d-60b912a37e76',
              'visible': True,
              'x': array([0.0, -0.026615150

In [ ]:
## piper双臂 双轨迹显示
# /liujinxin/code/tram/cosmos-predict2.5/outputs/piper_ros/piper_ros-noetic/src/piper_description/urdf/piper_description_v100.urdf

# -*- coding: utf-8 -*-
"""
Piper 双单臂（当作双臂系统）轨迹显示
- 加载两个 Piper 单臂 URDF
- 从 data.json 读取 joint (14 维)
- 前 7 维给左臂：6 arm + 1 gripper
- 后 7 维给右臂：6 arm + 1 gripper
- Plotly + PyBullet 做 FK 可视化
"""

import json
import os
import math
import shutil
import numpy as np
from pathlib import Path

import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import imageio.v2 as imageio


# ============================================================
# ✅ 相机参数
# ============================================================
CAMERAS = {
    "left_wrist":  dict(w=640,  h=480,  fx=450.0,  fy=450.0,  cx=None, cy=None),
    "right_wrist": dict(w=640,  h=480,  fx=450.0,  fy=450.0,  cx=None, cy=None),
    "head":        dict(w=1920, h=1080, fx=1100.0, fy=1100.0, cx=None, cy=None),
}

def cam_spec(name: str):
    s = dict(CAMERAS[name])
    if s.get("cx", None) is None:
        s["cx"] = s["w"] / 2.0
    if s.get("cy", None) is None:
        s["cy"] = s["h"] / 2.0
    return s

def cam_K(name: str):
    s = cam_spec(name)
    K = np.array([
        [s["fx"], 0.0,     s["cx"]],
        [0.0,     s["fy"], s["cy"]],
        [0.0,     0.0,     1.0]
    ], dtype=np.float64)
    return K

def cam_aspect(name: str):
    s = cam_spec(name)
    return float(s["w"]) / float(s["h"])


# ============================================================
# ✅ 路径参数
# ============================================================
urdf_path = "/liujinxin/code/tram/cosmos-predict2.5/outputs/piper_ros/piper_ros-noetic/src/piper_description/urdf/piper_description_v100.urdf"
data_path = Path("/liujinxin/dataset/piper/cloth_new/2026-03-12_demo_clothes_six/episode_0/data.json")

# 左右两台 Piper 的 base 摆放
# LEFT_BASE_POS  = [-0.22,  0.28, 0.0]
# RIGHT_BASE_POS = [-0.22, -0.28, 0.0]

# # 让两台机器人都朝向桌子中间
# LEFT_BASE_EULER  = [0.0, 0.0, 0.0]
# RIGHT_BASE_EULER = [0.0, 0.0, math.pi]

BASE_X = -0.22
BASE_Z = 0.0
ARM_GAP = 0.5  # 两台机械臂之间的横向距离

LEFT_BASE_POS  = [BASE_X,  ARM_GAP / 2.0, BASE_Z]
RIGHT_BASE_POS = [BASE_X, -ARM_GAP / 2.0, BASE_Z]

LEFT_BASE_EULER  = [0.0, 0.0, 0.0]
RIGHT_BASE_EULER = [0.0, 0.0, 0.0]

EXPORT_DIR = "export_fk_piper_bimanual"
FPS = 10
EXPORT_RENDER_CAM = "head"
SAVE_CAM_TXT = True
CAM_TXT_DIRNAME = "cameras"

CAM_OFFSET_M = np.array([0.05, 0.0, 0.035], dtype=np.float64)
CAM_PITCH_DOWN_DEG = -25.0

HEAD_POS = np.array([0.25, 0.0, 0.85], dtype=np.float64)
HEAD_PITCH_DOWN_DEG = -35.0
CAM_BOX_LWH = (0.06, 0.03, 0.03)


# ============================================================
# ✅ Piper joint 候选名
# 说明：
# 不同 URDF 版本命名可能不同，所以这里做自动匹配
# ============================================================
ARM_JOINT_NAME_CANDIDATES = [
    ["joint1", "Joint1", "shoulder_joint", "axis_joint_1"],
    ["joint2", "Joint2", "shoulder_lift_joint", "axis_joint_2"],
    ["joint3", "Joint3", "elbow_joint", "axis_joint_3"],
    ["joint4", "Joint4", "wrist_joint_1", "axis_joint_4"],
    ["joint5", "Joint5", "wrist_joint_2", "axis_joint_5"],
    ["joint6", "Joint6", "wrist_joint_3", "axis_joint_6"],
]

GRIPPER_CANDIDATE_NAMES = [
    "gripper_joint",
    "left_gripper_joint",
    "right_gripper_joint",
    "finger_joint",
    "finger_left_joint",
    "finger_right_joint",
    "gripper_axis",
    "joint7",
    "Joint7",
]

EE_CANDIDATE_NAMES = [
    "ee_link",
    "tool0",
    "tcp_link",
    "gripper_link",
    "end_effector",
    "link6",
    "Link6",
    "joint6",
    "Joint6",
]


# ============================================================
# ✅ 数据读取
# ============================================================
def read_json(fp: Path):
    with fp.open("r", encoding="utf-8") as f:
        return json.load(f)

def build_actions_from_data_json(fp: Path):
    rows = read_json(fp)
    if not isinstance(rows, list):
        raise ValueError(f"{fp} should be a list of dicts")

    left = []
    right = []
    tasks = []

    for i, d in enumerate(rows):
        if "joint" not in d:
            raise KeyError(f"row {i} missing key 'joint'")
        q = np.asarray(d["joint"], dtype=np.float64).reshape(-1)
        if q.shape[0] != 14:
            raise ValueError(f"row {i} joint dim should be 14, got {q.shape[0]}")

        left.append(q[:7])   # 6 arm + 1 gripper
        right.append(q[7:])  # 6 arm + 1 gripper
        tasks.append(d.get("task", []))

    left = np.asarray(left, dtype=np.float64)
    right = np.asarray(right, dtype=np.float64)
    # right = np.zeros_like(left)

    print("left actions shape :", left.shape)
    print("right actions shape:", right.shape)
    print("T =", len(left))
    return left, right, tasks


# ============================================================
# ✅ 数学/几何工具
# ============================================================
def rot_x(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([
        [1, 0, 0],
        [0, ca, -sa],
        [0, sa, ca]
    ], dtype=np.float64)

def rot_y(deg: float) -> np.ndarray:
    a = math.radians(deg)
    ca, sa = math.cos(a), math.sin(a)
    return np.array([
        [ ca, 0, sa],
        [  0, 1,  0],
        [-sa, 0, ca]
    ], dtype=np.float64)

def make_T(R: np.ndarray, t_xyz) -> np.ndarray:
    Tm = np.eye(4, dtype=np.float64)
    Tm[:3, :3] = R
    Tm[:3, 3] = np.array(t_xyz, dtype=np.float64)
    return Tm

def quat_xyzw_to_R(q_xyzw):
    return np.array(p.getMatrixFromQuaternion(q_xyzw), dtype=np.float64).reshape(3, 3)

def cam2world_from_link(points, R_list, link_idx, offset_m, pitch_down_deg):
    T_world_link = make_T(R_list[link_idx], points[link_idx])
    R_link_cam = rot_y(-pitch_down_deg)
    T_link_cam = make_T(R_link_cam, offset_m)
    return T_world_link @ T_link_cam

def head_cam2world_fixed():
    R = rot_y(-HEAD_PITCH_DOWN_DEG)
    return make_T(R, HEAD_POS)

def box_edges_from_T(T_world_obj, lwh):
    L, W, H = lwh
    x = L / 2
    y = W / 2
    z = H / 2

    corners_local = np.array([
        [ x,  y,  z],
        [ x,  y, -z],
        [ x, -y,  z],
        [ x, -y, -z],
        [-x,  y,  z],
        [-x,  y, -z],
        [-x, -y,  z],
        [-x, -y, -z],
    ], dtype=np.float64)

    R = T_world_obj[:3, :3]
    t = T_world_obj[:3, 3]
    corners_world = (R @ corners_local.T).T + t[None, :]

    edges = [
        (0,1),(0,2),(0,4),
        (3,1),(3,2),(3,7),
        (5,1),(5,4),(5,7),
        (6,2),(6,4),(6,7),
    ]
    xs, ys, zs = [], [], []
    for a, b in edges:
        pa, pb = corners_world[a], corners_world[b]
        xs += [pa[0], pb[0], None]
        ys += [pa[1], pb[1], None]
        zs += [pa[2], pb[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def axes_lines_from_T(T_world, axis_len=0.10):
    o = T_world[:3, 3]
    R = T_world[:3, :3]
    x_end = o + R[:, 0] * axis_len
    y_end = o + R[:, 1] * axis_len
    z_end = o + R[:, 2] * axis_len
    return o, x_end, y_end, z_end


# ============================================================
# ✅ PyBullet joint / link 工具
# ============================================================
def list_joints(robot_id: int):
    out = []
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        out.append(dict(
            jid=int(info[0]),
            jname=info[1].decode("utf-8"),
            jtype=int(info[2]),
            qidx=int(info[3]),
            uidx=int(info[4]),
            parent=int(info[16]),
            link_name=info[12].decode("utf-8"),
        ))
    return out

def name_to_joint_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        d[info[1].decode("utf-8")] = i
    return d

def name_to_link_index(robot_id: int):
    d = {}
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        d[info[12].decode("utf-8")] = i
        d[info[1].decode("utf-8")] = i
    return d

def get_revolute_prismatic_joint_indices(robot_id: int):
    valid_types = {p.JOINT_REVOLUTE, p.JOINT_PRISMATIC}
    indices = []
    names = []
    for i in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, i)
        jtype = int(info[2])
        if jtype in valid_types:
            indices.append(i)
            names.append(info[1].decode("utf-8"))
    return indices, names

def resolve_arm_joint_indices(robot_id: int):
    name2idx = name_to_joint_index(robot_id)
    resolved = []

    for cand_group in ARM_JOINT_NAME_CANDIDATES:
        found = None
        for n in cand_group:
            if n in name2idx:
                found = name2idx[n]
                break
        resolved.append(found)

    if all(x is not None for x in resolved):
        return resolved

    # 兜底：直接取前 6 个可运动关节
    movable_indices, movable_names = get_revolute_prismatic_joint_indices(robot_id)
    if len(movable_indices) < 6:
        raise RuntimeError(f"Expected at least 6 movable joints, got {len(movable_indices)}")
    print("[WARN] arm joint names not fully matched, fallback to first 6 movable joints:")
    print("       ", movable_names[:6])
    return movable_indices[:6]

def resolve_gripper_joint_indices(robot_id: int):
    name2idx = name_to_joint_index(robot_id)
    out = []
    for n in GRIPPER_CANDIDATE_NAMES:
        if n in name2idx:
            out.append(name2idx[n])
    # 去重
    out = list(dict.fromkeys(out))
    return out

def find_ee_link_index(robot_id: int):
    name2link = name_to_link_index(robot_id)
    for name in EE_CANDIDATE_NAMES:
        if name in name2link:
            return name2link[name], name

    # 兜底：最后一个 link
    last_idx = p.getNumJoints(robot_id) - 1
    info = p.getJointInfo(robot_id, last_idx)
    return last_idx, info[12].decode("utf-8")

def get_links_world(robot_id: int):
    num = p.getNumJoints(robot_id)
    points = np.zeros((num, 3), dtype=np.float64)
    parent = np.full((num,), -1, dtype=np.int32)
    R_list = np.zeros((num, 3, 3), dtype=np.float64)

    for i in range(num):
        info = p.getJointInfo(robot_id, i)
        parent[i] = info[16]
        st = p.getLinkState(robot_id, i, computeForwardKinematics=True)
        points[i] = np.array(st[4], dtype=np.float64)
        R_list[i] = quat_xyzw_to_R(st[5])

    return points, parent, R_list

def skeleton_lines(points, parent, mask=None):
    xs, ys, zs = [], [], []
    for i in range(len(points)):
        if mask is not None and (not mask[i]):
            continue
        pidx = parent[i]
        if pidx >= 0:
            if mask is not None and (not mask[pidx]):
                continue
            p1, p2 = points[pidx], points[i]
            xs += [p1[0], p2[0], None]
            ys += [p1[1], p2[1], None]
            zs += [p1[2], p2[2], None]
    return np.array(xs, dtype=object), np.array(ys, dtype=object), np.array(zs, dtype=object)

def scene_ranges(points_list, cams_T_dict):
    all_pts = list(points_list)
    for Tm in cams_T_dict.values():
        all_pts.append(Tm[:3, 3][None, :])
    all_pts = np.vstack(all_pts)
    mins = all_pts.min(axis=0)
    maxs = all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2.0
    return center, half


# ============================================================
# ✅ 关节设置
# ============================================================
def map_gripper_value_to_joint(v):
    """
    你的数据里 gripper 大概是 0~1.
    这里先线性映射到一个较小开合范围。
    后面如果你看到夹爪开合不对，只改这里即可。
    """
    # v = float(np.clip(v, 0.0, 1.0))
    # return 0.04 * (1.0 - v)
    
    v = float(np.clip(v, 0.0, 1.0))
    return 0.04 * v

def set_piper_from_action(robot_id: int, action_7: np.ndarray, arm_joint_indices, gripper_joint_indices):
    action_7 = np.asarray(action_7, dtype=np.float64).reshape(-1)
    if action_7.shape[0] != 7:
        raise ValueError(f"Expected action dim 7, got {action_7.shape[0]}")

    arm6 = action_7[:6]
    grip = action_7[6]

    for idx, q in zip(arm_joint_indices, arm6):
        p.resetJointState(robot_id, idx, float(q))

    if len(gripper_joint_indices) > 0:
        gq = map_gripper_value_to_joint(grip)
        for idx in gripper_joint_indices:
            try:
                p.resetJointState(robot_id, idx, gq)
            except Exception:
                pass

def compute_frame(robot_id: int, action_7, arm_joint_indices, gripper_joint_indices):
    set_piper_from_action(robot_id, action_7, arm_joint_indices, gripper_joint_indices)
    pts, parent, R = get_links_world(robot_id)
    return pts, parent, R


# ============================================================
# ✅ 初始化 PyBullet
# ============================================================
try:
    is_conn = p.getConnectionInfo().get("isConnected", 0)
except Exception:
    is_conn = 0

if not is_conn:
    p.connect(p.DIRECT)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

# 读动作
actions_left, actions_right, tasks = build_actions_from_data_json(data_path)
T = actions_left.shape[0]

# 加载左右两台 Piper
robot_id_left = p.loadURDF(
    urdf_path,
    basePosition=LEFT_BASE_POS,
    baseOrientation=p.getQuaternionFromEuler(LEFT_BASE_EULER),
    useFixedBase=True
)

robot_id_right = p.loadURDF(
    urdf_path,
    basePosition=RIGHT_BASE_POS,
    baseOrientation=p.getQuaternionFromEuler(RIGHT_BASE_EULER),
    useFixedBase=True
)

print("=== LEFT Piper joints ===")
for j in list_joints(robot_id_left):
    print(f"[{j['jid']:03d}] joint={j['jname']:<30s} link={j['link_name']:<30s} type={j['jtype']} parent={j['parent']}")

left_arm_joint_indices = resolve_arm_joint_indices(robot_id_left)
right_arm_joint_indices = resolve_arm_joint_indices(robot_id_right)

left_gripper_joint_indices = resolve_gripper_joint_indices(robot_id_left)
right_gripper_joint_indices = resolve_gripper_joint_indices(robot_id_right)

left_ee_link_idx, left_ee_link_name = find_ee_link_index(robot_id_left)
right_ee_link_idx, right_ee_link_name = find_ee_link_index(robot_id_right)

print("\n=== resolved joints ===")
print("left arm joint indices   :", left_arm_joint_indices)
print("right arm joint indices  :", right_arm_joint_indices)
print("left gripper joint idx   :", left_gripper_joint_indices)
print("right gripper joint idx  :", right_gripper_joint_indices)
print("left ee                  :", left_ee_link_idx, left_ee_link_name)
print("right ee                 :", right_ee_link_idx, right_ee_link_name)

T_world_head = head_cam2world_fixed()

mask_left = np.ones((p.getNumJoints(robot_id_left),), dtype=bool)
mask_right = np.ones((p.getNumJoints(robot_id_right),), dtype=bool)


# ============================================================
# ✅ 初始帧
# ============================================================
f0 = 0
pts_left, parent_left, R_left = compute_frame(
    robot_id_left, actions_left[f0], left_arm_joint_indices, left_gripper_joint_indices
)
pts_right, parent_right, R_right = compute_frame(
    robot_id_right, actions_right[f0], right_arm_joint_indices, right_gripper_joint_indices
)

def cams_T_from_frame(pts_left, R_left, left_ee_idx, pts_right, R_right, right_ee_idx):
    return {
        "left_wrist":  cam2world_from_link(pts_left,  R_left,  left_ee_idx,  CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "right_wrist": cam2world_from_link(pts_right, R_right, right_ee_idx, CAM_OFFSET_M, CAM_PITCH_DOWN_DEG),
        "head":        T_world_head,
    }

cams_T = cams_T_from_frame(
    pts_left, R_left, left_ee_link_idx,
    pts_right, R_right, right_ee_link_idx
)

x_left, y_left, z_left = skeleton_lines(pts_left, parent_left, mask=mask_left)
x_right, y_right, z_right = skeleton_lines(pts_right, parent_right, mask=mask_right)


# ============================================================
# ✅ Plotly FigureWidget
# ============================================================
fig = go.FigureWidget()

# 左臂：蓝
fig.add_trace(go.Scatter3d(
    x=pts_left[:, 0], y=pts_left[:, 1], z=pts_left[:, 2],
    mode="markers",
    marker=dict(size=3, color="royalblue"),
    name="left_points"
))
fig.add_trace(go.Scatter3d(
    x=x_left, y=y_left, z=z_left,
    mode="lines",
    line=dict(width=5, color="royalblue"),
    name="left_bones"
))

# 右臂：橙
fig.add_trace(go.Scatter3d(
    x=pts_right[:, 0], y=pts_right[:, 1], z=pts_right[:, 2],
    mode="markers",
    marker=dict(size=3, color="orangered"),
    name="right_points"
))
fig.add_trace(go.Scatter3d(
    x=x_right, y=y_right, z=z_right,
    mode="lines",
    line=dict(width=5, color="orangered"),
    name="right_bones"
))

def add_axes_traces(prefix, Tm, axis_len=0.15, visible=True):
    o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len)
    fig.add_trace(go.Scatter3d(
        x=[o[0], x_end[0]], y=[o[1], x_end[1]], z=[o[2], x_end[2]],
        mode="lines", line=dict(width=6, color="red"),
        name=f"{prefix}_x", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], y_end[0]], y=[o[1], y_end[1]], z=[o[2], y_end[2]],
        mode="lines", line=dict(width=6, color="green"),
        name=f"{prefix}_y", visible=visible, showlegend=False
    ))
    fig.add_trace(go.Scatter3d(
        x=[o[0], z_end[0]], y=[o[1], z_end[1]], z=[o[2], z_end[2]],
        mode="lines", line=dict(width=6, color="blue"),
        name=f"{prefix}_z", visible=visible, showlegend=False
    ))

def add_box_trace(prefix, Tm, lwh, visible=True):
    x, y, z = box_edges_from_T(Tm, lwh)
    fig.add_trace(go.Scatter3d(
        x=x, y=y, z=z,
        mode="lines",
        line=dict(width=4, color="purple"),
        name=f"{prefix}_box", visible=visible, showlegend=False
    ))

trace_map = {
    "left_points": [0],
    "left_bones": [1],
    "right_points": [2],
    "right_bones": [3],
    "world_axes": [],
    "cam_axes": {k: [] for k in CAMERAS.keys()},
    "cam_boxes": {k: [] for k in CAMERAS.keys()},
}

base_world = np.eye(4)
start_idx = len(fig.data)
add_axes_traces("world", base_world, axis_len=0.20, visible=True)
trace_map["world_axes"] = list(range(start_idx, start_idx + 3))

for cam_name in CAMERAS.keys():
    start_idx = len(fig.data)
    add_axes_traces(cam_name, cams_T[cam_name], axis_len=0.12, visible=True)
    trace_map["cam_axes"][cam_name] = list(range(start_idx, start_idx + 3))

    start_idx = len(fig.data)
    add_box_trace(cam_name, cams_T[cam_name], CAM_BOX_LWH, visible=True)
    trace_map["cam_boxes"][cam_name] = [start_idx]

center, half = scene_ranges([pts_left, pts_right], cams_T)
task_text = tasks[f0][0] if (len(tasks[f0]) > 0 and isinstance(tasks[f0], list)) else ""
fig.update_layout(
    title=(
        f"Piper Bimanual FK (Frame {f0}) | "
        f"left=blue, right=orange | "
        f"task={task_text}"
    ),
    margin=dict(l=0, r=0, t=40, b=0),
    height=750,
    scene=dict(
        xaxis=dict(range=[center[0]-half, center[0]+half], title="X"),
        yaxis=dict(range=[center[1]-half, center[1]+half], title="Y"),
        zaxis=dict(range=[center[2]-half, center[2]+half], title="Z"),
        aspectmode="cube",
    ),
    legend=dict(orientation="h"),
)


# ============================================================
# ✅ UI controls
# ============================================================
slider = widgets.IntSlider(value=f0, min=0, max=T-1, step=1, description="frame", continuous_update=False)

cb_show_left_points = widgets.Checkbox(value=True, description="left points")
cb_show_left_bones  = widgets.Checkbox(value=True, description="left bones")
cb_show_right_points = widgets.Checkbox(value=True, description="right points")
cb_show_right_bones  = widgets.Checkbox(value=True, description="right bones")

cb_show_world = widgets.Checkbox(value=True, description="World axes")
cb_show_cam_axes = widgets.Checkbox(value=True, description="Camera axes")
cb_show_cam_box  = widgets.Checkbox(value=True, description="Camera box")

cam_select = widgets.SelectMultiple(
    options=list(CAMERAS.keys()),
    value=tuple(CAMERAS.keys()),
    description="Cams",
    rows=len(CAMERAS)
)

dd_export = widgets.Dropdown(options=["none", "gif", "mp4"], value="none", description="Export")
btn_export = widgets.Button(description="Export", button_style="info")
out_log = widgets.Output()

ui_row1 = widgets.HBox([slider])
ui_row2 = widgets.HBox([
    cb_show_left_points, cb_show_left_bones,
    cb_show_right_points, cb_show_right_bones
])
ui_row3 = widgets.HBox([cb_show_world, cb_show_cam_axes, cb_show_cam_box])
ui_row4 = widgets.HBox([cam_select, dd_export, btn_export])

play = widgets.Play(
    interval=int(1000 / FPS),
    value=int(slider.value),
    min=0,
    max=T-1,
    step=1,
    description="Play",
    disabled=False,
)
widgets.jslink((play, "value"), (slider, "value"))

speed = widgets.IntSlider(value=FPS, min=1, max=60, step=1, description="fps", continuous_update=False)
cb_loop = widgets.Checkbox(value=False, description="Loop")

def _on_speed_change(change):
    play.interval = int(1000 / max(1, int(change["new"])))

speed.observe(_on_speed_change, names="value")

def _on_play_value(change):
    if not cb_loop.value:
        return
    if int(change["new"]) >= T - 1:
        play.value = 0

play.observe(_on_play_value, names="value")

ui_row_play = widgets.HBox([play, speed, cb_loop])


# ============================================================
# ✅ 显隐控制
# ============================================================
def set_visible(indices, v: bool):
    for idx in indices:
        fig.data[idx].visible = v

def apply_visibility():
    cams_on = set(cam_select.value)
    with fig.batch_update():
        set_visible(trace_map["left_points"], cb_show_left_points.value)
        set_visible(trace_map["left_bones"],  cb_show_left_bones.value)
        set_visible(trace_map["right_points"], cb_show_right_points.value)
        set_visible(trace_map["right_bones"],  cb_show_right_bones.value)
        set_visible(trace_map["world_axes"], cb_show_world.value)

        for cam in CAMERAS.keys():
            cam_enabled = cam in cams_on
            set_visible(trace_map["cam_axes"][cam], cb_show_cam_axes.value and cam_enabled)
            set_visible(trace_map["cam_boxes"][cam], cb_show_cam_box.value and cam_enabled)


# ============================================================
# ✅ 更新帧
# ============================================================
def update_frame(f):
    f = int(f)

    pts_left, parent_left, R_left = compute_frame(
        robot_id_left, actions_left[f], left_arm_joint_indices, left_gripper_joint_indices
    )
    pts_right, parent_right, R_right = compute_frame(
        robot_id_right, actions_right[f], right_arm_joint_indices, right_gripper_joint_indices
    )

    cams_T_new = cams_T_from_frame(
        pts_left, R_left, left_ee_link_idx,
        pts_right, R_right, right_ee_link_idx
    )
    center, half = scene_ranges([pts_left, pts_right], cams_T_new)

    x_left, y_left, z_left = skeleton_lines(pts_left, parent_left, mask=mask_left)
    x_right, y_right, z_right = skeleton_lines(pts_right, parent_right, mask=mask_right)

    task_text = tasks[f][0] if (len(tasks[f]) > 0 and isinstance(tasks[f], list)) else ""

    with fig.batch_update():
        fig.data[0].x = pts_left[:, 0]
        fig.data[0].y = pts_left[:, 1]
        fig.data[0].z = pts_left[:, 2]

        fig.data[1].x = x_left
        fig.data[1].y = y_left
        fig.data[1].z = z_left

        fig.data[2].x = pts_right[:, 0]
        fig.data[2].y = pts_right[:, 1]
        fig.data[2].z = pts_right[:, 2]

        fig.data[3].x = x_right
        fig.data[3].y = y_right
        fig.data[3].z = z_right

        idx = 4
        idx += 3  # world axes

        for cam in CAMERAS.keys():
            Tm = cams_T_new[cam]
            o, x_end, y_end, z_end = axes_lines_from_T(Tm, axis_len=0.12)

            fig.data[idx + 0].x = [o[0], x_end[0]]
            fig.data[idx + 0].y = [o[1], x_end[1]]
            fig.data[idx + 0].z = [o[2], x_end[2]]

            fig.data[idx + 1].x = [o[0], y_end[0]]
            fig.data[idx + 1].y = [o[1], y_end[1]]
            fig.data[idx + 1].z = [o[2], y_end[2]]

            fig.data[idx + 2].x = [o[0], z_end[0]]
            fig.data[idx + 2].y = [o[1], z_end[1]]
            fig.data[idx + 2].z = [o[2], z_end[2]]
            idx += 3

            bx, by, bz = box_edges_from_T(Tm, CAM_BOX_LWH)
            fig.data[idx].x = bx
            fig.data[idx].y = by
            fig.data[idx].z = bz
            idx += 1

        fig.layout.title = (
            f"Piper Bimanual FK (Frame {f}) | "
            f"left=blue, right=orange | "
            f"task={task_text}"
        )
        fig.layout.scene.xaxis.range = [center[0]-half, center[0]+half]
        fig.layout.scene.yaxis.range = [center[1]-half, center[1]+half]
        fig.layout.scene.zaxis.range = [center[2]-half, center[2]+half]

    apply_visibility()

def on_slider_change(change):
    update_frame(change["new"])

slider.observe(on_slider_change, names="value")

for w in [
    cb_show_left_points, cb_show_left_bones,
    cb_show_right_points, cb_show_right_bones,
    cb_show_world, cb_show_cam_axes, cb_show_cam_box, cam_select
]:
    w.observe(lambda c: apply_visibility(), names="value")

apply_visibility()


# ============================================================
# ✅ Matplotlib 渲染（导出用）
# ============================================================
def render_frame_matplotlib(f: int, render_cam: str = EXPORT_RENDER_CAM):
    pts_left, parent_left, R_left = compute_frame(
        robot_id_left, actions_left[f], left_arm_joint_indices, left_gripper_joint_indices
    )
    pts_right, parent_right, R_right = compute_frame(
        robot_id_right, actions_right[f], right_arm_joint_indices, right_gripper_joint_indices
    )

    cams_T = cams_T_from_frame(
        pts_left, R_left, left_ee_link_idx,
        pts_right, R_right, right_ee_link_idx
    )

    aspect = cam_aspect(render_cam)
    base_h = 6.0
    fig_m = plt.figure(figsize=(base_h * aspect, base_h))
    ax = fig_m.add_subplot(111, projection="3d")

    ax.scatter(pts_left[:, 0], pts_left[:, 1], pts_left[:, 2], s=10, c="royalblue")
    x1, y1, z1 = skeleton_lines(pts_left, parent_left, mask=mask_left)
    ax.plot(x1.astype(float), y1.astype(float), z1.astype(float), linewidth=2, c="royalblue")

    ax.scatter(pts_right[:, 0], pts_right[:, 1], pts_right[:, 2], s=10, c="orangered")
    x2, y2, z2 = skeleton_lines(pts_right, parent_right, mask=mask_right)
    ax.plot(x2.astype(float), y2.astype(float), z2.astype(float), linewidth=2, c="orangered")

    for Tm in cams_T.values():
        o = Tm[:3, 3]
        RR = Tm[:3, :3]
        ax.quiver(o[0], o[1], o[2], RR[0, 0], RR[1, 0], RR[2, 0], length=0.08, color="r")
        ax.quiver(o[0], o[1], o[2], RR[0, 1], RR[1, 1], RR[2, 1], length=0.08, color="g")
        ax.quiver(o[0], o[1], o[2], RR[0, 2], RR[1, 2], RR[2, 2], length=0.08, color="b")

    all_pts = np.vstack([pts_left, pts_right, np.stack([T[:3, 3] for T in cams_T.values()])])
    mins, maxs = all_pts.min(axis=0), all_pts.max(axis=0)
    center = (mins + maxs) / 2.0
    span = (maxs - mins).max()
    if span < 1e-6:
        span = 1.0
    half = span / 2.0

    task_text = tasks[f][0] if (len(tasks[f]) > 0 and isinstance(tasks[f], list)) else ""

    ax.set_xlim(center[0] - half, center[0] + half)
    ax.set_ylim(center[1] - half, center[1] + half)
    ax.set_zlim(center[2] - half, center[2] + half)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")
    ax.set_title(f"Piper Bimanual FK (Frame {f}) | task={task_text}")

    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_rgb(), dtype=np.uint8).reshape(h, w, 3)
    plt.close(fig_m)
    return img, cams_T


# ============================================================
# ✅ 保存相机 txt
# ============================================================
def maybe_save_camera_txt(cams_T: dict, frame_idx: int):
    if not SAVE_CAM_TXT:
        return

    cams_dir = os.path.join(EXPORT_DIR, CAM_TXT_DIRNAME)
    os.makedirs(cams_dir, exist_ok=True)

    for name in CAMERAS.keys():
        np.savetxt(os.path.join(cams_dir, f"intrinsic_{name}.txt"), cam_K(name), fmt="%.9f")

    for name, T_wc in cams_T.items():
        np.savetxt(os.path.join(cams_dir, f"extrinsic_{name}_f{frame_idx:06d}.txt"), T_wc, fmt="%.9f")


# ============================================================
# ✅ 导出 gif/mp4
# ============================================================
def export_frames(export_type: str):
    os.makedirs(EXPORT_DIR, exist_ok=True)

    with out_log:
        clear_output()
        print(f"[export] type={export_type}")
        print("[export] renderer = matplotlib(Agg)")
        print("[export] render_aspect_from =", EXPORT_RENDER_CAM, cam_spec(EXPORT_RENDER_CAM))

    if export_type == "gif":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.gif")
        frames = []
        for f in range(T):
            img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
            frames.append(img)
            maybe_save_camera_txt(cams_T, f)
            if f % 10 == 0:
                with out_log:
                    print(f"[export] frame {f}/{T-1}")
        imageio.mimsave(out_path, frames, duration=1.0 / FPS)
        with out_log:
            print(f"[OK] saved gif to {out_path}")

    elif export_type == "mp4":
        out_path = os.path.join(EXPORT_DIR, "fk_compare.mp4")
        writer = imageio.get_writer(out_path, fps=FPS)
        try:
            for f in range(T):
                img, cams_T = render_frame_matplotlib(f, EXPORT_RENDER_CAM)
                writer.append_data(img)
                maybe_save_camera_txt(cams_T, f)
                if f % 10 == 0:
                    with out_log:
                        print(f"[export] frame {f}/{T-1}")
        finally:
            writer.close()
        with out_log:
            print(f"[OK] saved mp4 to {out_path}")
    else:
        with out_log:
            print("[INFO] export type is none")

def on_export_clicked(_):
    export_frames(dd_export.value)

btn_export.on_click(on_export_clicked)


# ============================================================
# ✅ 显示 UI
# ============================================================
display(ui_row1)
display(ui_row2)
display(ui_row3)
display(ui_row4)
display(ui_row_play)
display(fig)
display(out_log)


left actions shape : (9057, 7)
right actions shape: (9057, 7)
T = 9057
=== LEFT Piper joints ===
[000] joint=joint1                         link=link1                          type=0 parent=-1
[001] joint=joint2                         link=link2                          type=0 parent=0
[002] joint=joint3                         link=link3                          type=0 parent=1
[003] joint=joint4                         link=link4                          type=0 parent=2
[004] joint=joint5                         link=link5                          type=0 parent=3
[005] joint=joint6                         link=link6                          type=0 parent=4
[006] joint=joint7                         link=link7                          type=1 parent=5
[007] joint=joint8                         link=link8                          type=1 parent=5

=== resolved joints ===
left arm joint indices   : [0, 1, 2, 3, 4, 5]
right arm joint indices  : [0, 1, 2, 3, 4, 5]
left gripper joint idx   

FigureWidget({
    'data': [{'marker': {'color': 'royalblue', 'size': 3},
              'mode': 'markers',
              'name': 'left_points',
              'type': 'scatter3d',
              'uid': 'f4d3cb6a-30a3-4cd3-8c85-054fbab817cf',
              'visible': True,
              'x': {'bdata': ('AAAAwPUozL8AAADA9SjMvwAAAGD3D+' ... 'CjV8W/AAAA4Nw9oL8AAADgK12hvw=='),
                    'dtype': 'f8'},
              'y': {'bdata': ('AAAAAAAA0D8AAAAAAADQPwAAAEAR38' ... 'DpWdA/AAAAICKbzD8AAACAj9jQPw=='),
                    'dtype': 'f8'},
              'z': {'bdata': ('AAAAgO18vz8AAACg7Xy/PwAAAMB5+c' ... 'AF5sk/AAAA4OLbxj8AAAAgp/LGPw=='),
                    'dtype': 'f8'}},
             {'line': {'color': 'royalblue', 'width': 5},
              'mode': 'lines',
              'name': 'left_bones',
              'type': 'scatter3d',
              'uid': 'f83944de-003c-4d3e-b2fa-35126b1bdf5a',
              'visible': True,
              'x': array([-0.2199999988079071, -0.219999998807

Output()

/liujinxin/conda3/envs/lerobot_r1_test_seg_GR00T-Dreams/bin/python


/liujinxin/conda3/envs/lerobot_r1_test_seg_GR00T-Dreams/bin/python


8.1.7


0.9.18
